## Business Analytics Internship Assignment: Company Data Analysis

This notebook is designed to analyze a large company dataset from Kaggle, focusing on Indian companies for business insights. It follows a structured approach, including data loading, cleaning, filtering, summary statistics, and exporting the refined data.

### 1. Setup and Data Download

First, we'll download the dataset using `kagglehub`. This ensures we're working with the latest version of the '17M+ Company Dataset'.

In [ ]:
import kagglehub
import pandas as pd

# Download the latest version of the dataset
path = kagglehub.dataset_download("mfrye0/bigpicture-company-dataset")

print(f"Path to dataset files: {path}")

100%|██████████| 608M/608M [00:05<00:00, 109MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mfrye0/bigpicture-company-dataset/versions/5


### 1.1. Download Indian States Dataset

To improve our data cleaning, we'll download a dataset containing a list of Indian states. This will help us infer the country code for companies with missing `country_code` but a known Indian state.

In [ ]:
import kagglehub
import pandas as pd
import os

# Download the Indian states dataset
indian_states_path = kagglehub.dataset_download("arjunaraoc/india-states")

print(f"Path to Indian states dataset files: {indian_states_path}")

# Load the Indian states CSV and extract state names
# Corrected filename based on previous directory listing
indian_states_csv_path = os.path.join(indian_states_path, 'india-states.csv')

try:
    indian_states_df = pd.read_csv(indian_states_csv_path)
    # Assuming the state names are in a column named 'subdivision_name', based on inspection
    indian_state_names = indian_states_df['subdivision_name'].str.upper().tolist()
    print(f"Loaded {len(indian_state_names)} Indian state names.")
    print("First 5 Indian states:", indian_state_names[:5])
except FileNotFoundError:
    print(f"Error: The file {indian_states_csv_path} was not found.")
except KeyError:
    print("Error: 'subdivision_name' column not found in the Indian states dataset. Please check the column name.")
except Exception as e:
    print(f"An error occurred while loading Indian states: {e}")

100%|██████████| 545/545 [00:00<00:00, 917kB/s]

Extracting files...
Path to Indian states dataset files: /root/.cache/kagglehub/datasets/arjunaraoc/india-states/versions/1
Loaded 36 Indian state names.
First 5 Indian states: ['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'CHHATTISGARH']


In [ ]:
import os

# List contents of the downloaded directory to find the actual CSV file
print(f"Contents of the Indian states dataset directory '{indian_states_path}':")
for root, dirs, files in os.walk(indian_states_path):
    for name in files:
        if name.endswith('.csv'):
            print(os.path.join(root, name))

Contents of the Indian states dataset directory '/root/.cache/kagglehub/datasets/arjunaraoc/india-states/versions/1':
/root/.cache/kagglehub/datasets/arjunaraoc/india-states/versions/1/india-states.csv


### 2. Load Data into Pandas DataFrame

Now, we'll load the `companies.csv` file from the downloaded path into a pandas DataFrame. This allows us to easily manipulate and analyze the data.

In [ ]:
# Construct the full path to the companies.csv file
# Corrected file name based on directory listing
csv_file_path = f"{path}/companies-2023-q4-sm.csv"

# Load the CSV into a pandas DataFrame
try:
    df = pd.read_csv(csv_file_path)
    print("Dataset loaded successfully!")
    # Display the first 5 rows to get a glimpse of the data
    display(df.head())
    # Display basic information about the DataFrame
    df.info()
except FileNotFoundError:
    print(f"Error: The file {csv_file_path} was not found. Please ensure the dataset is downloaded correctly.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

Dataset loaded successfully!


,handle,name,website,industry,size,type,founded,city,state,country_code
0,company/&-shayna-solution-partners,& Shayna solution partners,andshayna.com,retail,1-10,Partnership,NaN,NaN,NaN,NaN
1,company/'addinall-management-and-systems','Addinall Management and Systems',NaN,business consulting and services,NaN,NaN,NaN,NaN,NaN,NaN
2,company/'baden-regio'-gemeinden-region-baden-w...,"'Baden Regio', Gemeinden Region Baden-Wettingen",baden-regio.ch,NaN,NaN,NaN,NaN,Wettingen,Aargau,CH
3,company/'friends-with-rice'​-stichting-'vriend...,'Friends With Rice' / Stichting 'Vrienden Met ...,vriendenmetrijst.nl,philanthropic fundraising services,1-10,Nonprofit,2014.0,Lisse,South Holland,NL
4,company/'i-say'-supported-living-services-limited,'I SAY' SUPPORTED LIVING SERVICES LIMITED,NaN,consumer services,NaN,NaN,NaN,Rochester,England,GB


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17154017 entries, 0 to 17154016
Data columns (total 10 columns):
 #   Column        Dtype  
---  ------        -----  
 0   handle        object 
 1   name          object 
 2   website       object 
 3   industry      object 
 4   size          object 
 5   type          object 
 6   founded       float64
 7   city          object 
 8   state         object 
 9   country_code  object 
dtypes: float64(1), object(9)
memory usage: 1.3+ GB


In [ ]:
import os

# List contents of the downloaded directory to find the actual CSV file
print(f"Contents of the downloaded directory '{path}':")
for root, dirs, files in os.walk(path):
    for name in files:
        if name.endswith('.csv'):
            print(os.path.join(root, name))


Contents of the downloaded directory '/root/.cache/kagglehub/datasets/mfrye0/bigpicture-company-dataset/versions/5':
/root/.cache/kagglehub/datasets/mfrye0/bigpicture-company-dataset/versions/5/companies-2023-q4-sm.csv


### 3. Data Cleaning

Data cleaning is a crucial step to ensure the quality and reliability of our analysis. We will address potential issues such as duplicate entries and missing values. The goal is to make sure our dataset is consistent and ready for filtering.

In [ ]:
import pandas as pd
import os
import kagglehub

# Ensure df is defined before proceeding to data cleaning.
# This cell re-loads the dataframe if it's not found, which might happen
# if previous cells were not run in order or the session state was lost.
if 'df' not in globals():
    print("DataFrame 'df' not found in current scope. Attempting to reload it.")

    # Re-obtain the dataset path if it's not defined
    if 'path' not in globals():
        print("Kaggle dataset 'path' not found. Attempting to re-download dataset path.")
        # This assumes the dataset is already downloaded to the cache from previous execution
        try:
            path = kagglehub.dataset_download("mfrye0/bigpicture-company-dataset")
            print(f"Path to dataset files: {path}")
        except Exception as e:
            print(f"Error re-downloading Kaggle dataset path: {e}")
            path = None # Set path to None if download fails

    if path:
        # Reconstruct the csv_file_path using the known file name
        csv_file_path = os.path.join(path, "companies-2023-q4-sm.csv")

        try:
            df = pd.read_csv(csv_file_path)
            print("DataFrame 'df' reloaded successfully!")
        except FileNotFoundError:
            print(f"Error: The file {csv_file_path} was not found. Please ensure the dataset is downloaded correctly by running cell 'cca2bcc5' and 'a1ffddb9' again.")
        except Exception as e:
            print(f"An error occurred while reloading the dataset: {e}")
    else:
        print("Error: 'path' variable is not defined. Please run the data download cell (e.g., 'cca2bcc5') first.")
else:
    print("DataFrame 'df' is already defined.")

DataFrame 'df' is already defined.


In [ ]:
import pandas as pd # Ensure pandas is imported for DataFrame operations

# Function to perform data cleaning
def clean_data(dataframe, indian_state_names):
    # Display initial shape and number of duplicates
    print(f"Initial DataFrame shape: {dataframe.shape}")
    initial_duplicates = dataframe.duplicated().sum()
    print(f"Initial number of duplicate rows: {initial_duplicates}")

    # Remove duplicate rows
    df_cleaned = dataframe.drop_duplicates().copy()
    print(f"DataFrame shape after removing duplicates: {df_cleaned.shape}")
    print(f"Number of duplicate rows after removal: {df_cleaned.duplicated().sum()}\n")

    # Handle missing values
    # Display columns with missing values and their counts
    missing_values = df_cleaned.isnull().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
    print("Columns with missing values before handling:\n", missing_values)

    # Infer 'IND' for missing 'country_code' if 'state' matches an Indian state
    if 'country_code' in df_cleaned.columns and 'state' in df_cleaned.columns:
        print("Attempting to infer 'IND' for missing country codes based on Indian states...")
        # Convert state column to uppercase for consistent matching
        df_cleaned['state_upper'] = df_cleaned['state'].str.upper()

        # Identify rows where country_code is NaN and state matches an Indian state
        ind_state_mask = (df_cleaned['country_code'].isnull()) & \
                         (df_cleaned['state_upper'].isin(indian_state_names))

        # Fill 'country_code' with 'IND' for these rows
        df_cleaned.loc[ind_state_mask, 'country_code'] = 'IND'

        # Drop the temporary 'state_upper' column
        df_cleaned.drop(columns=['state_upper'], inplace=True)
        print(f"Inferred 'IND' for {ind_state_mask.sum()} records.")

    # For 'website', missing values might mean no active website. We'll keep them as is for now,
    # as our filtering step will specifically look for active websites.
    # For 'industry', we will fill with 'Unknown' to keep these records if they meet other criteria,
    # but our filtering step will then check if 'Unknown' is in suitable_industries.
    # For 'country_code', we'll drop rows where this is missing as it's critical for filtering Indian companies.
    df_cleaned.dropna(subset=['country_code'], inplace=True)
    print(f"DataFrame shape after dropping rows with remaining missing 'country_code': {df_cleaned.shape}\n")

    # Fill missing 'industry' with 'Unknown' to keep these records if they meet other criteria
    if 'industry' in df_cleaned.columns:
        df_cleaned['industry'].fillna('Unknown', inplace=True)

    print("Missing values after initial handling:\n", df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])

    return df_cleaned

# Apply the cleaning function
# Ensure indian_state_names and df are defined (from d3a34a21 and other data loading cells)
if 'indian_state_names' in globals() and 'df' in globals():
    df_cleaned = clean_data(df, indian_state_names)
elif 'indian_state_names' not in globals():
    print("Error: 'indian_state_names' is not defined. Please run the Indian states dataset loading cell (d3a34a21) first.")
    df_cleaned = pd.DataFrame() # Assign an empty DataFrame to prevent NameError
elif 'df' not in globals():
    print("Error: 'df' is not defined. Please run the main dataset loading cells (e.g., 'cca2bcc5', 'a1ffddb9', 'fd3834bd') first.")
    df_cleaned = pd.DataFrame() # Assign an empty DataFrame to prevent NameError
else:
    print("Pre-requisite data (df or indian_state_names) is missing. Skipping data cleaning.")
    df_cleaned = pd.DataFrame() # Ensure df_cleaned is always defined, even if empty.


Initial DataFrame shape: (17154017, 10)
Initial number of duplicate rows: 1
DataFrame shape after removing duplicates: (17154016, 10)
Number of duplicate rows after removal: 0

Columns with missing values before handling:
 founded         8650131
type            5575556
state           4959036
city            3599757
website         3511964
country_code    3141779
size            2867330
industry        1309255
name               5196
dtype: int64
Attempting to infer 'IND' for missing country codes based on Indian states...
Inferred 'IND' for 971 records.
DataFrame shape after dropping rows with remaining missing 'country_code': (14013208, 10)



/tmp/ipykernel_1362/516914383.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned['industry'].fillna('Unknown', inplace=True)


Missing values after initial handling:
 name          1297
website    2620409
size       2094260
type       4613818
founded    6512035
city        582713
state      1853763
dtype: int64


### 4. Filter Companies

This section filters the companies based on specific criteria relevant to your internship assignment. The filters are defined using variables, making them easy to modify without changing the core logic. We'll look for:

*   **Indian companies**: `country_code` is 'IND'.
*   **Active websites**: `website_url` is not missing and is a valid URL.
*   **Suitable industries**: Companies belonging to a predefined list of industries.
*   **Sufficient company information**: Ensuring key columns like `name` and `description` are not null.

In [ ]:
import numpy as np
import pandas as pd # Ensure pandas is imported for pd.isna()

# Function to filter the DataFrame based on specified criteria
def filter_companies(dataframe, target_country_code, suitable_industries):
    # Make a copy to avoid SettingWithCopyWarning
    df_filtered = dataframe.copy()

    print(f"Initial records before filtering: {df_filtered.shape[0]}")

    # Filter 1: Indian companies
    df_filtered = df_filtered[df_filtered['country_code'] == target_country_code]
    print(f"Records after filtering for '{target_country_code}' companies: {df_filtered.shape[0]}")

    # Filter 2: Companies with active websites
    # Assuming an 'active' website means the 'website' is not null/empty
    if not df_filtered.empty: # Add check for empty DataFrame
        df_filtered = df_filtered[df_filtered['website'].notna() & (df_filtered['website'] != '')]
    print(f"Records after filtering for active websites: {df_filtered.shape[0]}")

    # Filter 3: Companies in suitable industries
    # Using a helper function to check if any of the company's industries are in the suitable list
    def is_suitable_industry(industry_str, suitable_list):
        if pd.isna(industry_str) or industry_str == 'Unknown':
            return False
        # Convert both to uppercase for case-insensitive comparison
        # The 'industry' column contains single industry strings, not comma-separated lists.
        return industry_str.upper() in [s.upper() for s in suitable_list]

    # Apply the industry filter (using 'industry' column)
    if not df_filtered.empty: # Add check for empty DataFrame
        df_filtered = df_filtered[df_filtered['industry'].apply(lambda x: is_suitable_industry(x, suitable_industries))]
    print(f"Records after filtering for suitable industries: {df_filtered.shape[0]}")

    # Filter 4: Sufficient company information (e.g., name is not null)
    # Note: The dataset does not contain a 'description' column, so we filter only by 'name'.
    if not df_filtered.empty: # Add check for empty DataFrame
        df_filtered = df_filtered[df_filtered['name'].notna() & (df_filtered['name'] != '')]
    print(f"Records after filtering for sufficient information (name): {df_filtered.shape[0]}")

    return df_filtered

# --- Define your filter criteria here (easy to modify) ---
target_country_code = 'IND'
suitable_industries = [
    'Information Technology & Services',
    'Computer Software',
    'Financial Services',
    'Marketing and Advertising',
    'E-learning',
    'Health, Wellness and Fitness',
    'Renewables & Environment',
    'Biotechnology',
    'Food & Beverages',
    'Retail'
    # Add or remove industries as per your analysis needs
]

# Apply the filtering function
df_filtered = filter_companies(df_cleaned, target_country_code, suitable_industries)

print("\nFiltered data preview:")
display(df_filtered.head())

Initial records before filtering: 14013208
Records after filtering for 'IND' companies: 974
Records after filtering for active websites: 792
Records after filtering for suitable industries: 54
Records after filtering for sufficient information (name): 54

Filtered data preview:


,handle,name,website,industry,size,type,founded,city,state,country_code
199530,company/wizcaps,WIZCAPS,wizcaps.com,financial services,11-50,Privately Held,2017.0,Vashi,Maharashtra,IND
223964,company/avama-naturals,Avama Naturals,avamanaturals.com,retail,1-10,Privately Held,2017.0,Delhi,Delhi,IND
273602,company/finmates-business,FinMates Business Solutions Private Limited,finmates.in,financial services,1-10,Privately Held,2019.0,Malad,Maharashtra,IND
462072,company/decoding-credit,Decoding Credit,decodingcredit.in,financial services,1-10,Self-Owned,2019.0,Bangalore,Karnataka,IND
683960,company/falcon-invoice-discounting,Falcon Invoice Discounting,falconsgrup.com,financial services,51-200,Privately Held,2015.0,Gowra Fountainhead,Telangana,IND


### Debugging Filter: Inspecting Indian Companies

It appears that all Indian companies are being filtered out due to the 'active websites' criterion. Let's inspect the data for these companies before the website filter is applied to understand their `website` and `industry` values. This will help us refine our filtering logic if needed.

In [ ]:
# Get the DataFrame after only the country filter
df_indian_companies = df_cleaned[df_cleaned['country_code'] == target_country_code]

print(f"Details of {df_indian_companies.shape[0]} Indian companies before website filter:")
display(df_indian_companies[['name', 'website', 'industry', 'country_code']])

# Also check unique values and counts for websites and industries for these companies
print("\nWebsite values for Indian companies:")
print(df_indian_companies['website'].value_counts(dropna=False))

print("\nIndustry values for Indian companies:")
print(df_indian_companies['industry'].value_counts(dropna=False))

Details of 974 Indian companies before website filter:


,name,website,industry,country_code
21277,BDBP,bdbp.co,advertising services,IND
31898,CANADA Education,google.com,education administration programs,IND
37501,Chronicles Sporting Events Pvt. Ltd.,chroniclessports.com,spectator sports,IND
52509,Medicaid Ethos Private Limited,medicaidethos.com,international trade and development,IND
62859,Etablir Designs,etablirdesigns.com,architecture and planning,IND
...,...,...,...,...
17047683,SP Colour & Chemicals,spcolour.in,chemical manufacturing,IND
17061769,ETHICS GROUP OF COMPANIES,ethicsgroup.in,"transportation, logistics, supply chain and st...",IND
17074737,VVC.MOTORS,vvcmotors.co.in,motor vehicle manufacturing,IND
17116803,Isole Himalayas: The Himalayan DMC,isolehimalayas.com,travel arrangements,IND



Website values for Indian companies:
website
NaN                   182
youtube.com             5
facebook.com            5
instagram.com           5
linkedin.com            4
                     ... 
godaddy.com             1
messung.com             1
piaggio-cv.co.in        1
rdspvt.ltd              1
tirupatioffice.com      1
Name: count, Length: 770, dtype: int64

Industry values for Indian companies:
industry
it services and it consulting    72
Unknown                          45
advertising services             41
software development             35
real estate                      34
                                 ..
retail art dealers                1
government relations services     1
retail motor vehicles             1
outsourcing/offshoring            1
hotels and motels                 1
Name: count, Length: 169, dtype: int64


### 5. Display Summary Statistics

After cleaning and filtering the data, it's helpful to look at the summary statistics of our refined dataset. This gives us an overview of the numerical columns and helps in understanding the distribution and characteristics of the filtered companies.

In [ ]:
# Display general information about the filtered DataFrame
print("\n--- Information about Filtered DataFrame ---")
df_filtered.info()

# Display descriptive statistics for numerical columns
print("\n--- Descriptive Statistics for Numerical Columns ---")
display(df_filtered.describe())

# For categorical columns, we can look at value counts
print("\n--- Value Counts for Key Categorical Columns ---")
if 'country_code' in df_filtered.columns:
    print("Country Code Distribution:\n", df_filtered['country_code'].value_counts())
if 'industry' in df_filtered.columns: # Corrected from 'industries' to 'industry'
    # Display top 10 industries
    print("\nTop 10 Industries Distribution:\n", df_filtered['industry'].value_counts().head(10))


--- Information about Filtered DataFrame ---
<class 'pandas.core.frame.DataFrame'>
Index: 54 entries, 199530 to 17016662
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   handle        54 non-null     object 
 1   name          54 non-null     object 
 2   website       54 non-null     object 
 3   industry      54 non-null     object 
 4   size          54 non-null     object 
 5   type          53 non-null     object 
 6   founded       42 non-null     float64
 7   city          52 non-null     object 
 8   state         54 non-null     object 
 9   country_code  54 non-null     object 
dtypes: float64(1), object(9)
memory usage: 4.6+ KB

--- Descriptive Statistics for Numerical Columns ---


,founded
count,42.00000
mean,2012.97619
std,17.20676
min,1910.00000
25%,2012.25000
50%,2017.00000
75%,2019.00000
max,2022.00000



--- Value Counts for Key Categorical Columns ---
Country Code Distribution:
 country_code
IND    54
Name: count, dtype: int64

Top 10 Industries Distribution:
 industry
information technology & services    23
financial services                   15
retail                                6
food & beverages                      4
e-learning                            3
renewables & environment              2
biotechnology                         1
Name: count, dtype: int64


### 6. Export Filtered Companies to CSV

Finally, we will export the cleaned and filtered dataset to a new CSV file. This allows you to use the refined data for further analysis, reporting, or sharing.

In [ ]:
# Define the output file path
output_csv_path = 'filtered_indian_companies.csv'

# Export the filtered DataFrame to a CSV file
df_filtered.to_csv(output_csv_path, index=False)

print(f"Filtered companies successfully exported to '{output_csv_path}'")

# Optionally, display the first few rows of the exported data to confirm
print("\nPreview of the exported CSV file:")
exported_df = pd.read_csv(output_csv_path)
display(exported_df.head())

Filtered companies successfully exported to 'filtered_indian_companies.csv'

Preview of the exported CSV file:


,handle,name,website,industry,size,type,founded,city,state,country_code
0,company/wizcaps,WIZCAPS,wizcaps.com,financial services,11-50,Privately Held,2017.0,Vashi,Maharashtra,IND
1,company/avama-naturals,Avama Naturals,avamanaturals.com,retail,1-10,Privately Held,2017.0,Delhi,Delhi,IND
2,company/finmates-business,FinMates Business Solutions Private Limited,finmates.in,financial services,1-10,Privately Held,2019.0,Malad,Maharashtra,IND
3,company/decoding-credit,Decoding Credit,decodingcredit.in,financial services,1-10,Self-Owned,2019.0,Bangalore,Karnataka,IND
4,company/falcon-invoice-discounting,Falcon Invoice Discounting,falconsgrup.com,financial services,51-200,Privately Held,2015.0,Gowra Fountainhead,Telangana,IND


### 7. Preparing for Final Submission: Loading Filtered Data and Sample Output

Now that we have our `filtered_indian_companies.csv` (the 54 companies we shortlisted previously), we need to prepare it for your final assignment. This involves loading this data and also looking at a `sample-output.csv` to understand the exact format required for your submission. We'll ensure our final output matches this sample.

**Step 1: Load the filtered Indian companies data.**

In [ ]:
import pandas as pd

# Load the previously filtered Indian companies data
filtered_companies_path = 'filtered_indian_companies.csv'
try:
    df_final_process = pd.read_csv(filtered_companies_path)
    print(f"Successfully loaded {len(df_final_process)} filtered Indian companies.")
    display(df_final_process.head())
except FileNotFoundError:
    print(f"Error: '{filtered_companies_path}' not found. Please ensure the previous export step was successful.")
    df_final_process = pd.DataFrame() # Create an empty DataFrame to avoid errors


Successfully loaded 54 filtered Indian companies.


,handle,name,website,industry,size,type,founded,city,state,country_code
0,company/wizcaps,WIZCAPS,wizcaps.com,financial services,11-50,Privately Held,2017.0,Vashi,Maharashtra,IND
1,company/avama-naturals,Avama Naturals,avamanaturals.com,retail,1-10,Privately Held,2017.0,Delhi,Delhi,IND
2,company/finmates-business,FinMates Business Solutions Private Limited,finmates.in,financial services,1-10,Privately Held,2019.0,Malad,Maharashtra,IND
3,company/decoding-credit,Decoding Credit,decodingcredit.in,financial services,1-10,Self-Owned,2019.0,Bangalore,Karnataka,IND
4,company/falcon-invoice-discounting,Falcon Invoice Discounting,falconsgrup.com,financial services,51-200,Privately Held,2015.0,Gowra Fountainhead,Telangana,IND


### 7.1. Recalculate Completeness Score for `df_final_process`

The `completeness_score` was previously calculated on `df_filtered` before it was exported. Since `df_final_process` is a fresh load from `filtered_indian_companies.csv`, we need to recalculate this score to ensure it's available for the subsequent scoring steps. We also need to prepare the 'founded' column for score calculations.

In [ ]:
if not df_final_process.empty:
    # Define the same secondary columns that contribute to 'data completeness'
    secondary_columns_for_completeness = ['size', 'type', 'founded', 'city', 'state']

    # Calculate completeness score for each company in df_final_process
    # The score is the count of non-null values in the secondary_columns
    df_final_process['completeness_score'] = df_final_process[secondary_columns_for_completeness].notna().sum(axis=1)

    # Ensure 'founded' column is numeric for age score calculation later
    # Convert to numeric, coercing errors to NaN, then fill NaNs for calculation purposes if needed
    df_final_process['founded'] = pd.to_numeric(df_final_process['founded'], errors='coerce')

    print("Recalculated 'completeness_score' for df_final_process.")
    print("Preview of df_final_process with new 'completeness_score':")
    display(df_final_process[['name', 'completeness_score', 'founded']].head())
else:
    print("df_final_process is empty, skipping completeness score calculation.")

Recalculated 'completeness_score' for df_final_process.
Preview of df_final_process with new 'completeness_score':


,name,completeness_score,founded
0,WIZCAPS,5,2017.0
1,Avama Naturals,5,2017.0
2,FinMates Business Solutions Private Limited,5,2019.0
3,Decoding Credit,5,2019.0
4,Falcon Invoice Discounting,5,2015.0


**Step 2: Load the sample output CSV to understand the required format.**

This `sample-output.csv` acts as a template, showing us exactly what columns and format your final submission should have. We'll load it and display its columns.

In [ ]:
# Load the sample output CSV
sample_output_path = '/content/sample-output.csv' # Assuming the sample output is in the /content directory
try:
    df_sample_output = pd.read_csv(sample_output_path)
    print("Successfully loaded sample output CSV.")
    print("Sample output columns:")
    print(df_sample_output.columns.tolist())
    display(df_sample_output.head())
except FileNotFoundError:
    print(f"Error: '{sample_output_path}' not found. Please ensure the sample output file is uploaded or correctly path-specified.")
    df_sample_output = pd.DataFrame() # Create an empty DataFrame


Successfully loaded sample output CSV.
Sample output columns:
['Company Name', 'Website', 'City', 'Segment', 'What They Make', 'Revenue Band', 'Decision Maker', 'DM Title', 'DM Background', 'C1 Manufacturer Score', 'C1 Evidence', 'C2 India Score', 'C2 Evidence', 'C3 Differentiation Score', 'C3 Evidence', 'C4 Tech DM Score', 'C4 Evidence', 'C5 Tailwind Score', 'C5 Evidence', 'C6 Growth Score', 'C6 Evidence', 'Federer Score', 'Score Band', 'Verdict', 'Verdict Reasoning', 'Personalization Hook', 'Evidence Sources']


,Company Name,Website,City,Segment,What They Make,Revenue Band,Decision Maker,DM Title,DM Background,C1 Manufacturer Score,...,C5 Tailwind Score,C5 Evidence,C6 Growth Score,C6 Evidence,Federer Score,Score Band,Verdict,Verdict Reasoning,Personalization Hook,Evidence Sources
0,Ananth Technologies,ananthtech.com,Hyderabad,Defence electronics / aerospace,"Satellite systems, avionics, radar subsystems,...",Rs.100-300Cr,Dr. Subba Rao Pavuluri,Founder & Chairman,"Ex-ISRO scientist, PhD, built company from ISR...",Strong,...,Strong,Defence indigenization policy (Make-in-India d...,Strong,Revenue Rs.270Cr FY24 with +30% YoY growth. Rs...,100,A — Strong Federer,Strong Pass,Textbook Federer. Scientist-founder with gen-2...,First private Indian company authorized for GS...,ananthtech.com; MCA filings; BSE annual report...
1,Avantel Limited,avantel.in,Hyderabad,Defence electronics / communications,"Tactical communication systems, software-defin...",Rs.100-300Cr,Dr. Abburi Vidyasagar,Founder & MD,"IIT Kharagpur M.Tech, ex-HAL Hyderabad, 30+ ye...",Strong,...,Strong,"Defence indigenization, Make-in-India for mili...",Strong,Revenue Rs.248Cr FY25 — 3x growth from FY21. R...,100,A — Strong Federer,Strong Pass,"Scientist-founder from IIT-KGP + HAL, 3x reven...",Inaugurated a Rs.56Cr new facility in Hyderaba...,avantel.in; BSE filings FY24 and FY25; facilit...
2,Lokesh Machines,lokeshmachines.com,Hyderabad,Precision engineering / aerospace components,"CNC machine tools, multi-spindle heads, aerosp...",Rs.100-300Cr,M. Lokeswara Rao,Founder & CMD,"Ex-HMT, 46 years experience in machine tool ma...",Strong,...,Strong,Aerospace component manufacturing is China+1 b...,Strong,Market cap up 65% in 1 year. Aerospace divisio...,100,A — Strong Federer,Strong Pass,Family-led with strong gen-2 technical transit...,Gen-2 leader with NJIT M.S. heading a new Aero...,lokeshmachines.com; BSE annual report FY24; AS...
3,ABR Organics,abrorganics.net,Hyderabad,Specialty chemicals / advanced materials,Polyimide films and resins for space and defen...,<Rs.50Cr,Dr. K.V.C. Rao,Founder,Ex-ISRO scientist with PhD. Co-founder Dr. Din...,Strong,...,Moderate,Specialty materials for space/defence have tai...,Weak,Website still references ISO 9001:2008 (not up...,70,B — Probable Federer,Borderline,Exceptionally strong on differentiation (C3) a...,Co-founders are ex-ISRO and Lehigh PhD scienti...,abrorganics.net; DSIR recognition list; ISRO s...
4,Alkali Metals Limited,alkalimetals.com,Hyderabad,Custom synthesis / specialty chemicals,"Niche heterocyclic compounds — tetrazoles, pyr...",Rs.50-100Cr,Y.S.R. Venkata Rao,MD (Gen-2),"B.E. Mechanical, Fellow of Institution of Engi...",Strong,...,Moderate,Specialty chemicals benefit from China+1 and p...,Moderate,Revenue Rs.82Cr and growing. USFDA approval is...,80,A — Strong Federer,Pass,"Solid Federer profile. Gen-2 technical MD, nic...",56-year-old specialty chemicals company just r...,alkalimetals.com; BSE filings; USFDA facility ...


### 8. Initializing the Final Submission DataFrame and Column Mapping

Now, we'll create an empty DataFrame that has all the columns from the `sample-output.csv`. Then, we'll copy the matching information from our `df_final_process` into this new DataFrame. This ensures our final submission has the correct structure from the start.

In [ ]:
if not df_final_process.empty and not df_sample_output.empty:
    # Create a new DataFrame with the same columns as the sample output, filled with NaNs
    final_submission_df = pd.DataFrame(columns=df_sample_output.columns)

    # Map existing columns from df_final_process to final_submission_df
    # Note: Column names must match exactly (case-sensitive) or be mapped explicitly
    column_mapping = {
        'name': 'Company Name',
        'website': 'Website', # Using 'Website' as per sample output
        'city': 'City',
        'state': 'State',
        'industry': 'Industry/Segment'
    }

    # Populate the final_submission_df with data from df_final_process
    # We will iterate through each row of df_final_process and append to final_submission_df
    # This is not the most efficient for very large datasets, but clear for a beginner.
    current_index = 0
    for idx, row in df_final_process.iterrows():
        new_row = {}
        for original_col, final_col in column_mapping.items():
            if original_col in row.index:
                new_row[final_col] = row[original_col]
            else:
                new_row[final_col] = None # Or np.nan
        final_submission_df.loc[current_index] = new_row # Assign new_row to the current index
        current_index += 1

    print(f"Initialized final submission DataFrame with {len(final_submission_df)} entries.")
    print("Preview of the initialized DataFrame:")
    display(final_submission_df.head())
else:
    print("Skipping initialization as either filtered data or sample output is empty.")
    final_submission_df = pd.DataFrame(columns=df_sample_output.columns if not df_sample_output.empty else []) # Ensure it's defined


Initialized final submission DataFrame with 54 entries.
Preview of the initialized DataFrame:


,Company Name,Website,City,Segment,What They Make,Revenue Band,Decision Maker,DM Title,DM Background,C1 Manufacturer Score,...,C5 Tailwind Score,C5 Evidence,C6 Growth Score,C6 Evidence,Federer Score,Score Band,Verdict,Verdict Reasoning,Personalization Hook,Evidence Sources
0,WIZCAPS,wizcaps.com,Vashi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Avama Naturals,avamanaturals.com,Delhi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FinMates Business Solutions Private Limited,finmates.in,Malad,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Decoding Credit,decodingcredit.in,Bangalore,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Falcon Invoice Discounting,falconsgrup.com,Gowra Fountainhead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 9. Gathering Publicly Available Information (Web Scraping - **Advanced & Challenging**)

This is a more advanced and potentially complex step. We need to visit each company's official website to gather more details, such as their LinkedIn page and to identify a key decision-maker (like a Founder or CEO).

**Important Considerations for Web Scraping:**

*   **Website Structure Varies:** Every website is different. What works for one might not work for another. It's hard to create a single script that perfectly scrapes all information from all websites.
*   **Rate Limits & Blocking:** Websites might block automated access if too many requests are made too quickly. This can lead to your script being temporarily or permanently blocked.
*   **`robots.txt`:** Websites have a `robots.txt` file that tells web crawlers what they are allowed to access. It's good practice to respect these rules.
*   **CAPTCHAs:** Many sites use CAPTCHAs to prevent automated access.
*   **Privacy & Legality:** Always be mindful of privacy policies and the legality of scraping data.

Given these challenges, a fully automated and reliable solution for finding LinkedIn pages and decision-makers for all companies without specific APIs is **extremely difficult and often not feasible** for a general-purpose script. For an internship assignment, these are typically tasks that require some **manual research**.

**For this demonstration, I will provide a basic web scraping framework. It will attempt to:
1.  Check if a website is reachable.
2.  Look for common keywords on the page that might indicate a LinkedIn presence or contact information.
3.  Mark fields as 'Manual Review' if automated scraping is unsuccessful.**

We will use `requests` to fetch website content and `BeautifulSoup` to parse HTML. You might need to install these libraries first.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import time
import numpy as np

# Install necessary libraries if not already installed
# !pip install requests beautifulsoup4

def get_website_content(url, timeout=10):
    """Fetches the content of a given URL."""
    try:
        # Add a common User-Agent header to mimic a web browser
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, timeout=timeout, headers=headers)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        return response.text
    except requests.exceptions.RequestException as e:
        # print(f"Error fetching {url}: {e}")
        return None

def find_linkedin_from_website(website_url, soup):
    """Attempts to find a LinkedIn profile URL on the website's pages or in meta tags."""
    linkedin_url = None
    # Look for 'linkedin' in links
    for a_tag in soup.find_all('a', href=True):
        if 'linkedin.com' in a_tag['href']:
            linkedin_url = a_tag['href']
            # Ensure it's a full URL
            if not linkedin_url.startswith('http'):
                linkedin_url = urljoin(website_url, linkedin_url)
            return linkedin_url

    # Check for meta tags (less common for LinkedIn, but worth a try for general social links)
    for meta_tag in soup.find_all('meta', property=True):
        if 'linkedin.com' in str(meta_tag.get('content')):
            return meta_tag.get('content')

    # Fallback to general search (e.g., Google search for 'company name + linkedin') - this requires a separate API/approach
    # We'll mark for manual review for now
    return None

def extract_decision_maker_info(soup):
    """Attempts to extract decision maker information (very difficult to automate reliably)."""
    # This is highly heuristic and likely to fail for many sites.
    # Common places: 'About Us', 'Team', 'Leadership' pages.
    # For a general scraper, this is best left to manual review.

    # Example: Look for common titles in text content (highly unreliable)
    text = soup.get_text().lower()
    if 'founder' in text:
        return 'Founder (Manual Review)'
    if 'ceo' in text:
        return 'CEO (Manual Review)'
    if 'managing director' in text:
        return 'Managing Director (Manual Review)'

    return None

# Iterate through each company to gather info
if not final_submission_df.empty:
    final_submission_df['LinkedIn Page'] = np.nan
    final_submission_df['Decision Maker'] = np.nan
    final_submission_df['Evidence/Source Links'] = np.nan # To store links to where info was found
    final_submission_df['C1-Data Completeness Score (Internal)'] = np.nan # Initialize scoring columns
    final_submission_df['C2-Industry Score (Internal)'] = np.nan
    final_submission_df['C3-Website Score (Internal)'] = np.nan
    final_submission_df['C4-LinkedIn Score (Internal)'] = np.nan
    final_submission_df['C5-Decision Maker Score (Internal)'] = np.nan
    final_submission_df['C6-Company Age Score (Internal)'] = np.nan
    final_submission_df['Final Score (Internal)'] = np.nan
    final_submission_df['Qualified/Not Qualified'] = np.nan

    # Merge original df_final_process to get 'founded' and 'completeness_score' for scoring
    # Ensure we merge on a unique identifier if possible, for now using 'Company Name' as proxy assuming uniqueness
    df_final_process_copy = df_final_process.copy()
    df_final_process_copy['Company Name'] = df_final_process_copy['name'] # Align column name for merge
    # Only select relevant columns for merge that are not yet in final_submission_df
    merge_cols = ['Company Name', 'founded', 'completeness_score'] # 'completeness_score' was added earlier
    # Adjust for potential column name differences in final_submission_df if 'founded' is needed
    final_submission_df = pd.merge(
        final_submission_df,
        df_final_process_copy[merge_cols],
        on='Company Name',
        how='left',
        suffixes=('', '_original')
    )

    print("Starting web scraping process (this might take some time and might not find all info automatically)...")

    for index, row in final_submission_df.iterrows():
        company_name = row['Company Name']
        website = row['Website'] # Corrected from 'Official Website' to 'Website'

        if pd.notna(website) and website.strip() != '':
            # Ensure website has a scheme (http/https)
            if not website.startswith('http://') and not website.startswith('https://'):
                website = 'https://' + website # Assume HTTPS as default

            print(f"Processing {company_name} ({website})...")
            page_content = get_website_content(website)

            if page_content:
                soup = BeautifulSoup(page_content, 'html.parser')

                # Try to find LinkedIn page
                linkedin_url = find_linkedin_from_website(website, soup)
                if linkedin_url:
                    final_submission_df.at[index, 'LinkedIn Page'] = linkedin_url
                    final_submission_df.at[index, 'Evidence/Source Links'] = linkedin_url # Link to LinkedIn
                else:
                    final_submission_df.at[index, 'LinkedIn Page'] = 'Manual Review: Not found on site'

                # Try to extract decision maker info
                decision_maker = extract_decision_maker_info(soup)
                if decision_maker:
                    final_submission_df.at[index, 'Decision Maker'] = decision_maker
                    if pd.isna(final_submission_df.at[index, 'Evidence/Source Links']): # Only if not already set by LinkedIn
                        final_submission_df.at[index, 'Evidence/Source Links'] = website # Link to company website
                else:
                    final_submission_df.at[index, 'Decision Maker'] = 'Manual Review: Not found'
            else:
                final_submission_df.at[index, 'LinkedIn Page'] = 'Manual Review: Website unreachable or error'
                final_submission_df.at[index, 'Decision Maker'] = 'Manual Review: Website unreachable or error'
                final_submission_df.at[index, 'Evidence/Source Links'] = 'Website unreachable'
        else:
            final_submission_df.at[index, 'LinkedIn Page'] = 'Manual Review: No website listed'
            final_submission_df.at[index, 'Decision Maker'] = 'Manual Review: No website listed'
            final_submission_df.at[index, 'Evidence/Source Links'] = 'No website listed'

        # Add a small delay to avoid being blocked by websites
        time.sleep(0.5)

    print("Web scraping process completed. Review 'LinkedIn Page', 'Decision Maker', and 'Evidence/Source Links' columns for 'Manual Review' entries.")
    display(final_submission_df[['Company Name', 'Website', 'LinkedIn Page', 'Decision Maker', 'Evidence/Source Links']].head(10)) # Corrected column name
else:
    print("Cannot perform web scraping as 'final_submission_df' is empty.")

Starting web scraping process (this might take some time and might not find all info automatically)...
Processing WIZCAPS (https://wizcaps.com)...


/tmp/ipykernel_1362/2044405465.py:123: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Manual Review: Website unreachable or error' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_submission_df.at[index, 'LinkedIn Page'] = 'Manual Review: Website unreachable or error'
/tmp/ipykernel_1362/2044405465.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Manual Review: Website unreachable or error' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_submission_df.at[index, 'Decision Maker'] = 'Manual Review: Website unreachable or error'
/tmp/ipykernel_1362/2044405465.py:125: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Website unreachable' has dtype in

Processing Avama Naturals (https://avamanaturals.com)...
Processing FinMates Business Solutions Private Limited (https://finmates.in)...
Processing Decoding Credit (https://decodingcredit.in)...
Processing Falcon Invoice Discounting (https://falconsgrup.com)...
Processing Sënsiblë Wëalth (https://sensiblewealth.in)...
Processing Coimbatore Bharathi Biotech Private Limited (https://cbblimited.com)...
Processing Moneymax Fingrow Pvt.Ltd. (https://moneymaxfingrow.com)...
Processing Kashmir Box (https://kashmirbox.com)...
Processing ASDR Infotech Pvt. Ltd (https://asdrinfotech.com)...
Processing Naaima Embedded Technology (https://naaima.com)...
Processing The Founder Story (https://thefounderstory.com)...
Processing Skillslearn Academy (https://skillslearn.in)...
Processing TivonaGlobal Technologies (https://tivonaglobal.com)...
Processing Zingle Eats (https://sultantakeaway.com)...
Processing Selfieism technologies Pvt. Ltd. (https://selfieismtechsolutions.com)...
Processing Sygnovate IT

,Company Name,Website,LinkedIn Page,Decision Maker,Evidence/Source Links
0,WIZCAPS,wizcaps.com,Manual Review: Website unreachable or error,Manual Review: Website unreachable or error,Website unreachable
1,Avama Naturals,avamanaturals.com,Manual Review: Website unreachable or error,Manual Review: Website unreachable or error,Website unreachable
2,FinMates Business Solutions Private Limited,finmates.in,Manual Review: Not found on site,Founder (Manual Review),https://finmates.in
3,Decoding Credit,decodingcredit.in,Manual Review: Website unreachable or error,Manual Review: Website unreachable or error,Website unreachable
4,Falcon Invoice Discounting,falconsgrup.com,Manual Review: Not found on site,CEO (Manual Review),https://falconsgrup.com
5,Sënsiblë Wëalth,sensiblewealth.in,Manual Review: Website unreachable or error,Manual Review: Website unreachable or error,Website unreachable
6,Coimbatore Bharathi Biotech Private Limited,cbblimited.com,Manual Review: Website unreachable or error,Manual Review: Website unreachable or error,Website unreachable
7,Moneymax Fingrow Pvt.Ltd.,moneymaxfingrow.com,https://www.linkedin.com/company/moneymaxfingr...,Manual Review: Not found,https://www.linkedin.com/company/moneymaxfingr...
8,Kashmir Box,kashmirbox.com,Manual Review: Not found on site,Manual Review: Not found,NaN
9,ASDR Infotech Pvt. Ltd,asdrinfotech.com,https://www.linkedin.com/company/asdr-infotech...,Manual Review: Not found,https://www.linkedin.com/company/asdr-infotech...


### 10. Calculating Scoring Columns (C1-C6 and Final Score)

Since exact values for scoring cannot be determined automatically without specific external data or complex AI, we will use transparent, rule-based logic to estimate these scores. These scores are internal estimates based on the data we have and the (limited) information gathered from websites. You can adjust the scoring logic as per your internship guidelines.

**Scoring Rules (Examples - adjust as needed):**

*   **C1-Data Completeness Score:** Based on the `completeness_score` we calculated earlier (which considered `size`, `type`, `founded`, `city`, `state`). Higher completeness gets a higher score. (Max 5 points, matching previous score).
*   **C2-Industry Score:** All companies are in 'suitable industries', so we can assign a base score (e.g., 5 points).
*   **C3-Website Score:** All companies have active websites (filtered earlier), so a base score (e.g., 5 points).
*   **C4-LinkedIn Score:** 5 points if a LinkedIn page was found via scraping, 0 otherwise (or 2 for 'Manual Review').
*   **C5-Decision Maker Score:** 5 points if a decision maker was identified (even if 'Manual Review' was flagged, indicating a potential find), 0 otherwise.
*   **C6-Company Age Score:** Companies founded earlier might get higher scores. For instance, `founded` before 2010 = 5 points, 2010-2015 = 3 points, after 2015 = 1 point, Unknown = 0 points.

**Final Score:** A simple sum of C1-C6.

**Qualified/Not Qualified:** If `Final Score` is above a certain threshold (e.g., 20 points), the company is 'Qualified'.

In [ ]:
if not final_submission_df.empty:
    # C1-Data Completeness Score (max 5 points, directly from `completeness_score`)
    # The 'completeness_score' column was merged from df_final_process
    final_submission_df['C1-Data Completeness Score (Internal)'] = final_submission_df['completeness_score'].fillna(0)

    # C2-Industry Score (e.g., 5 points for all as they are in suitable industries)
    final_submission_df['C2-Industry Score (Internal)'] = 5

    # C3-Website Score (e.g., 5 points for all as they have active websites)
    final_submission_df['C3-Website Score (Internal)'] = 5

    # C4-LinkedIn Score (5 if found, 0 if not found or manual review)
    final_submission_df['C4-LinkedIn Score (Internal)'] = final_submission_df['LinkedIn Page'].apply(lambda x: 5 if pd.notna(x) and 'linkedin.com' in str(x) else 0)

    # C5-Decision Maker Score (5 if found/identified, 0 if not found)
    final_submission_df['C5-Decision Maker Score (Internal)'] = final_submission_df['Decision Maker'].apply(lambda x: 5 if pd.notna(x) and 'Manual Review' not in str(x) else 0)

    # C6-Company Age Score based on 'founded' year
    def calculate_company_age_score(founded_year):
        if pd.isna(founded_year):
            return 0
        year = int(founded_year)
        if year <= 2010:
            return 5 # Very established
        elif year <= 2015:
            return 3 # Established
        else:
            return 1 # Newer
    final_submission_df['C6-Company Age Score (Internal)'] = final_submission_df['founded'].apply(calculate_company_age_score)

    # Calculate Final Score (Sum of C1 to C6)
    final_submission_df['Final Score (Internal)'] = (
        final_submission_df['C1-Data Completeness Score (Internal)'] +
        final_submission_df['C2-Industry Score (Internal)'] +
        final_submission_df['C3-Website Score (Internal)'] +
        final_submission_df['C4-LinkedIn Score (Internal)'] +
        final_submission_df['C5-Decision Maker Score (Internal)'] +
        final_submission_df['C6-Company Age Score (Internal)']
    )

    # Mark Qualified/Not Qualified based on Final Score
    qualification_threshold = 20 # Example threshold, adjust as needed
    final_submission_df['Qualified/Not Qualified'] = final_submission_df['Final Score (Internal)'].apply(
        lambda score: 'Qualified' if score >= qualification_threshold else 'Not Qualified'
    )

    print("Scoring completed. Preview of scores:")
    display(final_submission_df[['Company Name', 'C1-Data Completeness Score (Internal)', 'C2-Industry Score (Internal)', 'C3-Website Score (Internal)', 'C4-LinkedIn Score (Internal)', 'C5-Decision Maker Score (Internal)', 'C6-Company Age Score (Internal)', 'Final Score (Internal)', 'Qualified/Not Qualified']].head(10))
else:
    print("Skipping scoring as 'final_submission_df' is empty.")

Scoring completed. Preview of scores:


,Company Name,C1-Data Completeness Score (Internal),C2-Industry Score (Internal),C3-Website Score (Internal),C4-LinkedIn Score (Internal),C5-Decision Maker Score (Internal),C6-Company Age Score (Internal),Final Score (Internal),Qualified/Not Qualified
0,WIZCAPS,5,5,5,0,0,1,16,Not Qualified
1,Avama Naturals,5,5,5,0,0,1,16,Not Qualified
2,FinMates Business Solutions Private Limited,5,5,5,0,0,1,16,Not Qualified
3,Decoding Credit,5,5,5,0,0,1,16,Not Qualified
4,Falcon Invoice Discounting,5,5,5,0,0,3,18,Not Qualified
5,Sënsiblë Wëalth,5,5,5,0,0,1,16,Not Qualified
6,Coimbatore Bharathi Biotech Private Limited,5,5,5,0,0,1,16,Not Qualified
7,Moneymax Fingrow Pvt.Ltd.,5,5,5,5,0,5,25,Qualified
8,Kashmir Box,5,5,5,0,0,3,18,Not Qualified
9,ASDR Infotech Pvt. Ltd,5,5,5,5,0,1,21,Qualified


### 11. Producing Final CSV and Exporting

Finally, we will prepare our DataFrame to exactly match the column names and order of the `sample-output.csv` and then save it as `final_submission.csv`. Any columns from the sample output that we couldn't automatically populate will be left as blank or marked for 'Manual Review'.

In [ ]:
if not final_submission_df.empty and not df_sample_output.empty:
    # Reorder and select columns to match df_sample_output
    final_columns = df_sample_output.columns.tolist()

    # Ensure all final_columns exist in final_submission_df, if not, add them as NaN
    for col in final_columns:
        if col not in final_submission_df.columns:
            final_submission_df[col] = np.nan

    # Select and reorder the columns
    df_export = final_submission_df[final_columns].copy()

    # Fill any remaining NaN values with a placeholder for manual review if appropriate
    # For string columns, replacing NaN with 'Manual Review' or empty string makes sense
    for col in df_export.columns:
        if df_export[col].dtype == 'object': # Check if column is of object type (usually strings)
            df_export[col] = df_export[col].fillna('Manual Review')
        else:
            df_export[col] = df_export[col].fillna('') # Fill numerical NaNs with empty string or 0 if appropriate

    # Define the output file path
    final_output_csv_path = 'final_submission.csv'

    # Export the DataFrame to a CSV file
    df_export.to_csv(final_output_csv_path, index=False)

    print(f"Final submission successfully exported to '{final_output_csv_path}'")
    print("Preview of the final exported CSV file (first 5 rows and all columns):")
    display(df_export.head())
else:
    print("Cannot export final submission as 'final_submission_df' is empty.")


Final submission successfully exported to 'final_submission.csv'
Preview of the final exported CSV file (first 5 rows and all columns):


,Company Name,Website,City,Segment,What They Make,Revenue Band,Decision Maker,DM Title,DM Background,C1 Manufacturer Score,...,C5 Tailwind Score,C5 Evidence,C6 Growth Score,C6 Evidence,Federer Score,Score Band,Verdict,Verdict Reasoning,Personalization Hook,Evidence Sources
0,WIZCAPS,wizcaps.com,Vashi,,,,Manual Review: Website unreachable or error,,,,...,,,,,,,,,,
1,Avama Naturals,avamanaturals.com,Delhi,,,,Manual Review: Website unreachable or error,,,,...,,,,,,,,,,
2,FinMates Business Solutions Private Limited,finmates.in,Malad,,,,Founder (Manual Review),,,,...,,,,,,,,,,
3,Decoding Credit,decodingcredit.in,Bangalore,,,,Manual Review: Website unreachable or error,,,,...,,,,,,,,,,
4,Falcon Invoice Discounting,falconsgrup.com,Gowra Fountainhead,,,,CEO (Manual Review),,,,...,,,,,,,,,,


In [ ]:
import csv

NA = "N/A"

# Each entry keyed by row index (0-53), values are dict of fields to fill
data = {}

def row(idx, **kwargs):
    data[idx] = kwargs

# 0 WIZCAPS
row(0,
    Segment="Fintech / Financial Services",
    WhatTheyMake="Online invoice discounting / bill discounting marketplace connecting SMEs seeking working capital with investors",
    RevenueBand=NA,
    DecisionMaker="Rajeev Ramachandran",
    DMTitle="Founder & CEO",
    DMBackground="Founded Wizcaps in 2016/2017; entrepreneur in SME lending/invoice financing space",
    C1=2, C1E="Pure fintech marketplace/services platform, no manufacturing or hardware component",
    C2=10, C2E="Headquartered in Vashi, Navi Mumbai, India; CIN U74999MH2017PTC298260 registered with RoC Mumbai",
    C3=4, C3E="Niche invoice-discounting marketplace model but highly commoditized vs. 50+ competitors (Tracxn lists 63 competitors); no patents or certifications found",
    C4=7, C4E="Founder & CEO Rajeev Ramachandran identifiable as primary decision maker (Tracxn, ZaubaCorp)",
    C5=6, C5E="Invoice discounting/alternative lending is a growing segment in Indian fintech, though crowded with 50+ competitors",
    C6=2, C6E="Tracxn reports Wizcaps has raised no funding rounds and ranks 47th of 57 competitors; no recent growth news found",
    Verdict="No",
    VerdictReasoning="Small, unfunded fintech marketplace ranked low among many competitors with no recent growth signals; low differentiation and low manufacturer/hardware fit.",
    Hook="Noticed Wizcaps has been bridging SME working-capital gaps via invoice discounting since 2017 — curious how you're scaling credit screening as deal volume grows.",
    Sources="Official website (wizcaps.com); LinkedIn company page; Tracxn; ZaubaCorp; Justdial"
)

# 1 Avama Naturals
row(1,
    Segment=NA, WhatTheyMake=NA, RevenueBand=NA,
    DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=NA, C1E="Unable to verify company - website unreachable and no independent sources found",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="Company website (avamanaturals.com) is unreachable and no independent, reliable source (LinkedIn, Crunchbase, Tracxn, news, registry) could confirm this company's identity or operations. Only a 2022 LinkedIn mention referencing 'Avama Naturals' as a partner of an unrelated blog project was found, insufficient for verification.",
    Hook=NA,
    Sources="Manual web search attempted; no reliable sources found beyond an unrelated LinkedIn post reference"
)

# 2 FinMates Business Solutions Private Limited
row(2,
    Segment="Financial Services / B2B Professional Services",
    WhatTheyMake="CFO advisory, accounting/F&A outsourcing, taxation, M&A and SME IPO listing advisory services",
    RevenueBand=NA,
    DecisionMaker="Pinkesh Jain",
    DMTitle="Founder & Lead CFO Service",
    DMBackground="Chartered Accountant (CA) and Company Secretary (CS); 17 years corporate experience; ex-PwC, ex-Neo Sports Broadcast, ex-Toyota/Lakozy Toyota, ex-Vibgyor, ex-Ampersand Group",
    C1=2, C1E="Pure professional/financial services firm (CFO advisory, accounting outsourcing) — no manufacturing or product component",
    C2=10, C2E="Headquartered in Malad West, Mumbai, Maharashtra, India",
    C3=5, C3E="Differentiated by founder's 16+ years combined team expertise and tech/digital/Web3 client specialization, but the virtual-CFO category is increasingly common in India",
    C4=4, C4E="Founder Pinkesh Jain is a CA/finance background leader, not a tech-titled decision maker (no CTO/VP Eng identified)",
    C5=6, C5E="Virtual CFO / outsourced finance services market is growing in India alongside SME and startup fundraising activity",
    C6=5, C6E="LinkedIn shows active recent posting (2025) and growing client base messaging, but no funding, headcount, or named growth metrics independently verified",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, verifiable professional services firm with a clearly identified founder, but pure-services model with no manufacturing/tech-product angle limits fit against a manufacturer-focused scoring rubric.",
    Hook="Saw FinMates has been helping IT/Web3 startups build out financial systems — would love to share how we support fast-scaling finance teams with the same kind of automation.",
    Sources="Official website (finmates.in); LinkedIn (Pinkesh Jain profile, FinMates company page); Justdial; Tracxn"
)

# 3 Decoding Credit
row(3,
    Segment="Financial Services (Loan/Credit Advisory)",
    WhatTheyMake="Credit and loan advisory services including term loans, bridge loans, non-fund based limits, and credit rating advisory for businesses",
    RevenueBand=NA,
    DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Pure financial advisory/brokerage service, no manufacturing component",
    C2=8, C2E="Operates as an Indian loan advisory business (decodingcredit.com); city given as Bangalore in source data, though exact registered HQ not independently confirmed",
    C3=3, C3E="Generic loan syndication/advisory offering with no unique IP or certification found; differs in name from very similar Athena Advisors 'Decoding Credit' research report series, indicating possible naming overlap/confusion",
    C4=NA, C4E="No named decision maker could be verified via LinkedIn, Crunchbase, or Tracxn",
    C5=6, C5E="Business loan/credit advisory demand is growing alongside India's MSME credit gap, a sector with broad tailwinds",
    C6=NA, C6E="No funding, hiring, or growth news found",
    Verdict="No",
    VerdictReasoning="No identifiable decision maker and minimal independent verification beyond the company's own website; generic loan brokerage with no differentiation.",
    Hook=NA,
    Sources="Company website (decodingcredit.com); general web search (no LinkedIn/Crunchbase/Tracxn profile found)"
)

# 4 Falcon Invoice Discounting  -- FRAUD FLAG
row(4,
    Segment="Fintech / Invoice Discounting (ALLEGED FRAUD / PONZI SCHEME)",
    WhatTheyMake="Marketed as a P2P invoice discounting investment platform (invoices from blue-chip companies); per CID Telangana investigation, the 'deals' were largely fabricated",
    RevenueBand=NA,
    DecisionMaker="Amardeep Kumar",
    DMTitle="Founder & Managing Director (also named in active criminal investigation)",
    DMBackground="Founded Falcon Invoice Discounting in 2015 under registered entity Capital Protection Force Pvt. Ltd.; named by Telangana CID as a key absconding accused in a confirmed ₹4,215 crore investment fraud (CEO Yogendra Singh arrested May 2025; Amardeep Kumar reportedly fled to Dubai)",
    C1=2, C1E="Marketed as a fintech investment platform, not a manufacturer; underlying business legitimacy itself is in serious doubt",
    C2=10, C2E="Headquartered in Hyderabad, Telangana, India",
    C3=1, C3E="No legitimate differentiation found; independent due-diligence blog (ALT Investor) found the company's incorporation documents (MoA) do not even list invoice discounting as a registered business activity",
    C4=3, C4E="Founder/MD identifiable, but is a named absconding accused in an active fraud investigation rather than a legitimate technology or business decision-maker",
    C5=NA, C5E="Not applicable — underlying business is confirmed fraudulent",
    C6=0, C6E="Company has stopped repaying investors since January 2025; Telangana CID has confirmed a ₹4,215 crore Ponzi-style fraud affecting over 7,000 investors, with multiple arrests in 2025",
    Verdict="No",
    VerdictReasoning="Confirmed law-enforcement investigation (Telangana CID) into Falcon Invoice Discounting as a large-scale Ponzi/investment fraud, with the founder/MD an absconding accused. This is not a legitimate sales prospect under any circumstance — outreach should be avoided entirely.",
    Hook=NA,
    Sources="Tracxn; LinkedIn company page; ALT Investor blog (independent due-diligence article, Jan 2024); Republic World news report (May 2025, CID Telangana arrest); Trustindex customer reviews documenting repayment defaults"
)

# 5 Sensible Wealth
row(5,
    Segment="Wealth Management / Financial Advisory",
    WhatTheyMake=NA,
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Wealth/financial advisory services, no manufacturing component",
    C2=8, C2E="Listed as headquartered in New Delhi, India per LinkedIn company page",
    C3=NA, C3E="Unable to verify any differentiating details beyond a bare LinkedIn listing",
    C4=NA, C4E="No named decision maker found",
    C5=6, C5E="Wealth management/financial advisory is a growing sector in India given rising HNI/retail investor base",
    C6=NA, C6E="No growth, funding, or news signals found",
    Verdict="No",
    VerdictReasoning="Only a bare LinkedIn listing could be verified; no website content, news, or named leadership found to support outreach personalization or scoring confidence.",
    Hook=NA,
    Sources="LinkedIn company page (in.linkedin.com/company/sensiblewealth); RocketReach email-format listing"
)

# 6 Coimbatore Bharathi Biotech
row(6,
    Segment="Biotechnology / Life Sciences R&D",
    WhatTheyMake="Agricultural biotechnology, biofertilizers, pharmaceuticals, marine biotechnology, environmental/ecological remediation technologies",
    RevenueBand=NA,
    DecisionMaker="Dr. Rahman",
    DMTitle="Technical/Scientific Lead (team page identifies him as a senior scientific figure)",
    DMBackground="Doctorate in Environmental Microbiology (Bharathiar University); MPhil Environmental Science (Anna University); Advanced Biomanufacturing of Biopharmaceuticals course at University of Cambridge (UK); former Associate Professor at University of Portsmouth and Teesside University; founded TeeGene Biotech and Tara Biologics; visiting professor at SOA University India",
    C1=9, C1E="Genuine biotechnology R&D and manufacturing company spanning pharmaceuticals, agri-biotech, and marine biotech applications",
    C2=10, C2E="Headquartered in Coimbatore/Mahalingapuram, Tamil Nadu, India",
    C3=8, C3E="Strong scientific differentiation: R&D spans plastic-degrading enzymes, agricultural waste-derived soap, sustainable water treatment, with an internationally credentialed scientific lead and global research collaborations (UK, US, China, UAE, New Zealand)",
    C4=6, C4E="Technical/scientific leadership clearly identifiable (Dr. Rahman) though exact CEO/founder title not confirmed",
    C5=8, C5E="Biotech, sustainable agriculture, and environmental remediation are strong global and Indian growth tailwinds",
    C6=NA, C6E="No specific recent funding, hiring, or expansion news found",
    Verdict="Yes",
    VerdictReasoning="Genuine R&D-driven biotech manufacturer with strong scientific credentials and differentiation across multiple high-growth verticals (agri-biotech, pharma, environmental remediation); strong fit for manufacturer/innovation-focused outreach.",
    Hook="Your team's work on plastic-degrading enzymes and sustainable agri-waste-to-soap conversion stood out — would love to explore how that R&D pipeline could scale with the right operational tooling.",
    Sources="Official website (cbblimited.com, team page); LinkedIn company page"
)

# 7 Moneymax Fingrow
row(7,
    Segment="Financial Services (Loan & Insurance Consulting)",
    WhatTheyMake="Financial consulting covering unsecured business loans, loan against property, debt syndication, home loans, and various insurance products",
    RevenueBand=NA,
    DecisionMaker="Dhejo and Mony",
    DMTitle="Co-Founders",
    DMBackground="19 years of combined experience across Banking and NBFC sectors (Sales, Credit, Operations, Field investigation) prior to founding Moneymax in 2017",
    C1=2, C1E="Pure financial consulting/brokerage, no manufacturing component",
    C2=10, C2E="Headquartered in Mahalingapuram, Chennai, Tamil Nadu, India",
    C3=4, C3E="68+ employees, 100+ channel partners, 300+ clients across various sectors; differentiated mainly by relationship-driven service model rather than unique IP",
    C4=4, C4E="Co-founders identifiable but backgrounds are in banking/NBFC operations, not technology",
    C5=6, C5E="Loan consulting/financial intermediation in South India continues to see steady demand",
    C6=5, C6E="51-200 employee LinkedIn listing with active recent posts (property loans, debt syndication offers) suggesting ongoing operations, though no specific recent funding or expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Stable, established financial consulting business (since 2004/2017) with named founders and a real employee base, but pure-services model with limited technology/manufacturing relevance.",
    Hook="Noticed Moneymax has built a strong channel-partner network of 100+ partners across South India for loan syndication — curious how your team manages partner-level reporting at that scale.",
    Sources="Official website (moneymaxfingrow.com, About Us page); LinkedIn company page; ZoomInfo"
)

# 8 Kashmir Box
row(8,
    Segment="E-commerce / Handicrafts & Artisanal Products Marketplace",
    WhatTheyMake="Online marketplace for Kashmiri handicrafts, handlooms, Pashmina shawls, carpets, dry fruits, spices, and gourmet foods (brand Koshur)",
    RevenueBand="₹5.62 Cr (FY2024, per Tracxn)",
    DecisionMaker="Muheet Mehraj",
    DMTitle="Co-Founder & CEO",
    DMBackground="Co-founded Kashmir Box in 2011 with Kashif Ahmad Khan to create a global marketplace for Kashmiri artisans; has grown the company through multiple angel funding rounds since 2017",
    C1=6, C1E="Product/marketplace company connecting artisans/manufacturers of handicrafts to a global market, rather than a direct manufacturer itself, though it has its own retail store/production-adjacent operations",
    C2=10, C2E="Headquartered in Srinagar, Jammu & Kashmir, India",
    C3=8, C3E="Strong differentiation: works with 5,000-10,000+ artisans, social-impact royalty/profit-sharing model, established Koshur gourmet food brand, and serves 40+ countries internationally",
    C4=8, C4E="CEO and Co-Founder Muheet Mehraj clearly and consistently identified across multiple sources (Tracxn, Owler, YourStory, The Dollar Business)",
    C5=7, C5E="Global handicrafts/artisanal e-commerce and social-impact retail are growing categories, particularly with rising demand for ethically-sourced products",
    C6=8, C6E="Raised a Seed round in October 2025 (total $236K across 4 rounds since 2017); historically reported 400% CAGR in early growth years; has 11 investors including 8 angels",
    Verdict="Yes",
    VerdictReasoning="Well-documented, actively funded company (latest round Oct 2025) with a clearly identified, long-tenured CEO, strong social-impact differentiation, and demonstrated multi-year growth trajectory.",
    Hook="Congrats on closing your Seed round in October 2025 — would love to learn how Kashmir Box plans to use the new capital to scale your artisan network further.",
    Sources="Tracxn; Owler; YourStory (2015 founder interview); The Dollar Business (2016 interview); Kashmir Box official website (About Us); IndiaMART"
)

# 9 ASDR Infotech
row(9,
    Segment="IT Services / SAP Consulting & Training",
    WhatTheyMake="SAP ERP consulting (implementation, upgrades, managed services), IT training, and digital marketing/web & app development services",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=4, C1E="IT services and training company; not a manufacturer though it serves manufacturing/automotive/EC&O client industries",
    C2=10, C2E="Headquartered in Pune, Maharashtra, India (LinkedIn/Facebook listing; CSV city 'Achal' likely a sub-locality)",
    C3=4, C3E="Partnership with Sydler Technologies for a SAP Center of Excellence; otherwise a fairly standard SAP consulting/training offering",
    C4=NA, C4E="No named founder/CEO could be verified",
    C5=6, C5E="SAP/ERP consulting and upskilling demand continues to grow with India's enterprise digital transformation push",
    C6=4, C6E="LinkedIn shows active hiring posts (Education Sales Executive, Full Stack Developer roles) suggesting steady operations, but no funding or major expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, active small IT services/training firm with steady hiring activity, but no identifiable senior decision-maker and limited differentiation beyond standard SAP consulting.",
    Hook="Saw ASDR Infotech is running an active SAP-focused training and internship program — would be glad to share how we help IT training providers scale program delivery.",
    Sources="Official website (asdrinfotech.com, asdrinfotech.in); LinkedIn company page; Facebook page"
)

# 10 Naaima Embedded Technology
row(10,
    Segment="Embedded Systems / Telecom & IT Hardware-Software",
    WhatTheyMake="Embedded systems engineering for telecom (VoIP, signalling protocols), set-top boxes, defense, and automation/testing solutions",
    RevenueBand=NA,
    DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=9, C1E="Genuine embedded hardware/software engineering company spanning telecom, defense, and automation hardware — strong manufacturer/hardware classification",
    C2=10, C2E="Headquartered in Tolichowki, Hyderabad, Telangana, India (CIN U32109TG2011PTC074918)",
    C3=6, C3E="Notable historical work developing a 'Smart Set-top box' and VoIP/signalling protocol solutions across Telecom, Scientific, and Defence domains, though company appears no longer active",
    C4=NA, C4E="Contact email (arif.adoni@naaima.com) suggests a person named Arif Adoni associated with the company, but no confirmed title or current role verified",
    C5=7, C5E="Embedded systems, IoT, and telecom hardware remain growth areas, though this specific company shows no current activity",
    C6=NA, C6E="Multiple independent corporate registries (Tofler, AllIndiaITR) list the company status as 'Struck Off' by the Registrar of Companies, indicating the company is no longer legally active",
    Verdict="No",
    VerdictReasoning="Despite a strong historical technical profile in embedded/telecom hardware, official MCA records confirm the company has been struck off (deregistered) and is no longer active — not a viable sales target.",
    Hook=NA,
    Sources="ZoomInfo; Tofler (company status); AllIndiaITR (company status); ClearTax company registry lookup"
)

# 11 The Founder Story
row(11,
    Segment="Media / Startup Content Platform",
    WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="If operational, would be a media/content platform, not a manufacturer",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="Could not independently verify thefounderstory.com as a distinct, currently operating company; no LinkedIn, Crunchbase, Tracxn, or news profile found matching this exact name/domain (several similarly-named but unrelated 'Founder Story'/'Startup Story' media platforms exist).",
    Hook=NA,
    Sources="General web search only; no verifiable company profile found"
)

# 12 Skillslearn Academy
row(12,
    Segment="Education / Skills Training (CLOSED)",
    WhatTheyMake="Personality development, language/communication skills, and rural/urban skill-development training programs",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Education/training services company, no manufacturing component",
    C2=10, C2E="Headquartered in Bangalore, Karnataka, India",
    C3=3, C3E="Focus on rural/urban skill-development and employability training, but no unique IP or certification found",
    C4=NA, C4E="No named founder/CEO verified",
    C5=5, C5E="Vocational/skills training is a steady growth area in India, though this specific provider appears defunct",
    C6=0, C6E="Justdial explicitly lists the business as 'Skillslearn Academy (Closed Down)'",
    Verdict="No",
    VerdictReasoning="Independent local business directory (Justdial) confirms this location/business has closed down; not a viable outreach target.",
    Hook=NA,
    Sources="Justdial (listing marked 'Closed Down'); LinkedIn company page; Internshala; Facebook page"
)

# 13 TivonaGlobal Technologies
row(13,
    Segment="IT Services / Cloud & DevOps Consulting",
    WhatTheyMake="Cloud computing, DevOps automation, infrastructure modernization, and CI/CD consulting services on AWS, Google Cloud, and Microsoft Azure",
    RevenueBand="$6 million (2026, per RocketReach)",
    DecisionMaker="Dharma Prakash",
    DMTitle="Co-Founder & Chief Executive Officer",
    DMBackground="Co-founded TivonaGlobal in mid-2017 alongside Co-Founder & CTO Murthy Manthena, building a cloud/DevOps consultancy partnered with AWS and Google Cloud",
    C1=4, C1E="Cloud/DevOps IT services and consulting company, not a hardware/physical manufacturer, though product-oriented in software infrastructure delivery",
    C2=10, C2E="Headquartered in Alwarpet, Chennai, Tamil Nadu, India",
    C3=6, C3E="Google Cloud Partner and Hashicorp System Integrator partner status; positions itself as a specialist cloud/DevOps transformation consultancy rather than a generalist IT shop",
    C4=9, C4E="Both Co-Founder & CEO (Dharma Prakash) and Co-Founder & CTO (Murthy Manthena) are clearly identified — strong tech leadership visibility",
    C5=8, C5E="Cloud migration, DevOps, and infrastructure modernization remain high-growth areas across Indian and global enterprise IT",
    C6=5, C6E="60 employees and $6M revenue reported (RocketReach); active hiring (Technical Lead role posted 2025); no major funding events found",
    Verdict="Yes",
    VerdictReasoning="Clearly identified technical co-founders (CEO and CTO), genuine cloud/DevOps specialization with recognized partner certifications (AWS, Google Cloud, Hashicorp), and demonstrated $6M revenue scale make this a strong outreach candidate, especially given clear tech decision-maker access.",
    Hook="Noticed TivonaGlobal is a certified Google Cloud and Hashicorp partner driving infrastructure-as-code adoption for Indian enterprises — would love to share how we support DevOps teams scaling CI/CD pipelines.",
    Sources="LinkedIn company page; ZoomInfo; RocketReach; SignalHire (Murthy Raju Manthena profile); Built In Chennai"
)

# 14 Zingle Eats
row(14,
    Segment="Food & Beverage / Quick Service Restaurant (QSR)",
    WhatTheyMake="Halal biryani and Indian quick-service restaurant food (brand: Sultan Biryani / Zingle Eats), operating in Chennai",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=3, C1E="Restaurant/QSR food preparation business; some light food production but not an industrial manufacturer",
    C2=10, C2E="Headquartered in Madipakkam/Nanganallur/Puzhuthivakkam, Chennai, Tamil Nadu, India",
    C3=3, C3E="Standard local QSR/biryani restaurant chain with multiple outlet listings; no unique IP, patents, or certifications found",
    C4=NA, C4E="No named founder/owner verified",
    C5=4, C5E="QSR/food delivery sector in India continues steady growth, though this is a small local operator, not a differentiated tech-enabled brand",
    C6=NA, C6E="No funding, expansion, or notable growth news found beyond standard customer reviews",
    Verdict="No",
    VerdictReasoning="Small local QSR/restaurant business with no technology angle, no identifiable decision maker, and no manufacturing or differentiation signal — poor fit for a B2B technology or manufacturing-focused sales motion.",
    Hook=NA,
    Sources="Zomato; Facebook page; Justdial; Yappe.in; hotel/restaurant directory listings"
)

# 15 Selfieism technologies Pvt. Ltd. (struck off)
row(15,
    Segment="IT Services / Web Design Training (STRUCK OFF)",
    WhatTheyMake="Web/app design training (HTML, CSS, PHP, WordPress, Unity3D) and website development services",
    RevenueBand=NA,
    DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=3, C1E="Small web design/training services shop, no manufacturing component",
    C2=10, C2E="Registered in Jabalpur, Madhya Pradesh, India (matches 'Rampur' sub-locality in source data)",
    C3=2, C3E="Standard local web design training offering with no unique differentiation found",
    C4=NA, C4E="Directors Prabhat Kumar Dubey and Om Prakash Tiwari listed in MCA records, but no current operating leadership confirmed",
    C5=4, C5E="Web design/IT training demand continues in tier-2 Indian cities, though irrelevant given company status",
    C6=0, C6E="Multiple independent corporate registries (Falcon Ebiz, AllIndiaITR, Tofler) confirm company status as 'Strike Off' by the Registrar of Companies",
    Verdict="No",
    VerdictReasoning="Official MCA registry records confirm the company has been struck off and is no longer an active legal entity — not viable for outreach.",
    Hook=NA,
    Sources="Falcon Ebiz (company registry); AllIndiaITR; Tofler; ClearTax registry lookup; Justdial"
)

# 16 Sygnovate IT Solutions
row(16,
    Segment="Electronic Security Systems / IT Hardware Distribution",
    WhatTheyMake="Electronic security products (CCTV surveillance, fire alarm, access control, video door phones, gate automation, home automation) plus IT/accounting software services",
    RevenueBand="$400,000 (2025, per RocketReach)",
    DecisionMaker="Praveen Patel",
    DMTitle="Director / Managing Director",
    DMBackground="MBA in Marketing (Vinayaka Mission's Research Foundation University); prior experience at Speck Systems Ltd before leading Sygnovate",
    C1=7, C1E="Distributor and systems-integrator of electronic security hardware (CCTV, fire alarm, access control) — closer to a product/hardware distribution business than pure services",
    C2=10, C2E="Headquartered in Jeedimetla, Hyderabad, Telangana, India",
    C3=5, C3E="One-stop-shop positioning across multiple security hardware categories (fire, CCTV, access control, gate automation, home automation), but largely a reseller/integrator of established manufacturer products rather than own-IP products",
    C4=6, C4E="Director/MD Praveen Patel identifiable as primary decision maker across multiple sources (RocketReach, ZoomInfo, Facebook)",
    C5=6, C5E="Electronic security and home/building automation demand continues to grow in India alongside smart-building adoption",
    C6=NA, C6E="No specific recent funding, hiring, or expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, identifiable hardware distribution/integration business in a growing security-tech category, but moderate differentiation since most products are sourced from other manufacturers rather than developed in-house.",
    Hook="Noticed Sygnovate covers everything from fire alarms to home automation under one roof — curious how you manage multi-vendor hardware integration at scale across your installs.",
    Sources="Company website (sygnovate.com); RocketReach; ZoomInfo; Facebook page; Dun & Bradstreet; IndiaMART"
)

# 17 Vitriv Technologies
row(17,
    Segment="IT Services",
    WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=4, C1E="Listed as an IT services company per LinkedIn (2-10 employees), no manufacturing details verifiable",
    C2=10, C2E="Headquartered in Mysuru (Ramakrishnagar), Karnataka, India per LinkedIn",
    C3=NA, C3E="Unable to verify specific products/services beyond basic LinkedIn industry listing",
    C4=NA, C4E="No named founder/decision maker verified despite source data flag",
    C5=5, C5E="General IT services sector tailwind in tier-2 Indian cities",
    C6=NA, C6E="No funding, hiring, or growth news found",
    Verdict="No",
    VerdictReasoning="Only a bare-bones LinkedIn listing (company size, founding year, location) could be verified; no website content, leadership names, or business description found to support confident scoring or personalization.",
    Hook=NA,
    Sources="LinkedIn company page (de.linkedin.com/company/vitriv-technologies); Indeed salary listing"
)

# 18 Intellect Training & Consulting Solutions
row(18,
    Segment="Education / Corporate Training",
    WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="If operational, likely a training/consulting services business based on name, not a manufacturer",
    C2=NA, C2E="Unable to independently verify; multiple unrelated 'Intellect'-named entities found in Bangalore (Intellect Design Arena, Intellect Education, Intellect Academy, Intellect Consultancy) but none confirmed to match 'intellecttech.in' specifically",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="The company name is too generic to reliably distinguish from several unrelated same-named entities in Bangalore; no source could confirm the specific company tied to domain 'intellecttech.in', consistent with the source data's 'Website unreachable or error' note.",
    Hook=NA,
    Sources="General web search only; multiple unrelated same-named entities found but none verified as a match"
)

# 19 Malghan Business Solutions
row(19,
    Segment="IT Services / Software Publishing & Consultancy",
    WhatTheyMake="Custom software development, IT consultancy, web design, and software publishing/supply services",
    RevenueBand=NA,
    DecisionMaker="Deepak Shivashankar Malghan",
    DMTitle="Director",
    DMBackground="Director of Malghan Business Solutions since its 2014 incorporation; company has been operating in Bangalore's software/IT consultancy sector for over a decade",
    C1=4, C1E="Software development and IT consultancy business, not a hardware/physical manufacturer",
    C2=10, C2E="Headquartered in Basaveshwara Nagar / Malleswaram, Bangalore, Karnataka, India",
    C3=3, C3E="Standard custom software development/consultancy offering with no unique IP, patents, or certifications found",
    C4=5, C4E="Director Deepak Shivashankar Malghan is identifiable via official MCA registry records (ZaubaCorp)",
    C5=6, C5E="General software/IT consultancy demand remains steady in Bangalore's tech ecosystem",
    C6=NA, C6E="No specific recent funding, hiring, or expansion news found; ZoomInfo notes 'very low activity levels' compared to sector peers",
    Verdict="Maybe",
    VerdictReasoning="Established, identifiable, decade-plus-old IT consultancy with a confirmed director, but limited differentiation and low recent growth signal.",
    Hook=NA,
    Sources="ZaubaCorp (company registry, director details); ZoomInfo; Justdial; AllIndiaITR; Apollo.io; StockKnocks"
)

# 20 ProSales-india
row(20,
    Segment=NA, WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=NA, C1E="Unable to verify",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="The domain prosales.co.in (Bangalore) could not be independently verified through any source; the only similarly-named entities found (Prosales Financial Services Pvt Ltd, a Mumbai-based NBFC subsidiary of Andromeda Loans; and ProSales.tech, a Malaysian/Singaporean proptech SaaS) appear to be unrelated companies. This confirms the source data's 'Website unreachable or error' finding.",
    Hook=NA,
    Sources="General web search only; unrelated similarly-named entities found but none verified as a match"
)

# 21 E-Team Informatica India Pvt. Ltd
row(21,
    Segment="IT Services / Software Development (Subsidiary of Italian Group)",
    WhatTheyMake="Website/web application development, ERP solutions (Microsoft Navision), BPO services, and control & automation software development",
    RevenueBand=NA,
    DecisionMaker="Anoop Nair",
    DMTitle="General Manager, E-Team Informatica",
    DMBackground="Senior operations leader for the Indian subsidiary of Italy-headquartered Gruppo Zenit; company recently underwent a strategic acquisition/integration into GruppoZenit India",
    C1=5, C1E="Software/IT services subsidiary of an Italian hardware-and-software group (Gruppo Zenit), with ERP and infrastructure-related services",
    C2=10, C2E="Headquartered at Technopark, Thiruvananthapuram, Kerala, India since 2005, as a subsidiary of Italy's Gruppo Zenit",
    C3=6, C3E="25-year track record (founded 2001) as the dedicated Indian delivery center for a multinational European group, spanning cloud, SAP Basis, disaster recovery, and digital security services",
    C4=6, C4E="General Manager Anoop Nair and other named leaders (Lorenzo Bozzola - Head Software Development, Luca Bianchi - Head of IT Services) identified via LinkedIn",
    C5=7, C5E="Cloud infrastructure, ERP, and digital security services continue to see strong enterprise demand, particularly for Indo-European delivery models",
    C6=8, C6E="In July 2025, GruppoZenit India Private Limited formally acquired/integrated E-Team Informatica's business operations, clients, and 25 years of workforce/expertise — a clear, recent, significant growth/M&A event",
    Verdict="Yes",
    VerdictReasoning="Long-established (25-year) IT services operation with a clear, very recent (July 2025) acquisition/integration into a larger multinational group structure, named leadership, and growing service scope (cloud, SAP Basis, digital security) — a strong and timely outreach opportunity.",
    Hook="Congrats on GruppoZenit India's July 2025 integration of E-Team Informatica's 25 years of Technopark expertise — would love to explore how the combined team is scaling cloud and SAP Basis delivery.",
    Sources="Technopark.in company directory; LinkedIn (E-Team Informatica, Gruppo Zenit pages); Apollo.io; Clodura.AI; SignalHire"
)

# 22 FMCG Ontime Jobs
row(22,
    Segment="HR / Staffing & Recruitment Services",
    WhatTheyMake="Manpower recruitment and HR consultancy services, with a specialization in FMCG sales-role placements (ASM, TSO, Sales Officer roles)",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Pure HR/staffing services business, no manufacturing component",
    C2=10, C2E="Headquartered in Bangalore, Karnataka, India",
    C3=4, C3E="16+ years of operating history (founded 2007/2008) with specialization across FMCG, F&B, QSR, real estate, and e-commerce/IT product recruitment verticals (also operating as 'OnTime Global')",
    C4=NA, C4E="No named founder/CEO verified",
    C5=5, C5E="Recruitment/staffing demand remains steady, particularly in FMCG and retail sectors",
    C6=4, C6E="Active job postings found for FMCG sales roles (ASM, TSO) indicating ongoing client demand; 4.2/5 Glassdoor rating with 78% recommend rate suggests stable operations",
    Verdict="Maybe",
    VerdictReasoning="Established, multi-vertical staffing firm with steady operations and positive employee reviews, but a pure-services HR model with no identifiable senior decision-maker limits fit.",
    Hook=NA,
    Sources="LinkedIn company page (On Time Solutions); Glassdoor; Joblum job listings; Company website (ontimesolutions.in, ontimeglobal.in)"
)

# 23 MICHI'S (JRJ Foods)
row(23,
    Segment="Manufacturing / FMCG Food & Confectionery",
    WhatTheyMake="Candies, toffees, and toffee bars (brand: MICHI'S) including Badam Toffee Bar and flavored candies; also operates a contract manufacturing unit for Parle Products (PARLE-G brand)",
    RevenueBand="₹18.6 Cr (FY ending Mar 2025, per Tracxn)",
    DecisionMaker="Jagdish Thakkar",
    DMTitle="Co-Founder & Managing Director (also Joint Managing Director per company website)",
    DMBackground="Born 1949, grew the company from its founding in 1978 (modern brand MICHI'S launched 2005); 45+ years of experience in confectionery manufacturing, marketing, and distribution; active life member of AMA (Ahmedabad Management Association)",
    C1=10, C1E="Genuine FMCG confectionery manufacturer operating two production units in Gujarat, including a contract manufacturing unit for Parle Products' PARLE-G brand",
    C2=10, C2E="Headquartered in GIDC Chhatral, Gandhinagar district, Gujarat, India",
    C3=7, C3E="Established distribution network spanning Gujarat, Rajasthan, Madhya Pradesh, and Chhattisgarh; exports to SAARC countries and the African region; Best Quality Award from PPPL; 51,000 sq ft constructed manufacturing facility",
    C4=4, C4E="Founder/MD Jagdish Thakkar identified, but background is in sales/marketing/manufacturing leadership rather than technology",
    C5=7, C5E="FMCG confectionery is a stable, growing category in India with rising rural and export demand",
    C6=6, C6E="₹18.6 Cr annual revenue with 68 employees (slightly declining 1% YoY per Tracxn); long-standing Parle contract manufacturing relationship provides revenue stability",
    Verdict="Yes",
    VerdictReasoning="Genuine, well-established FMCG manufacturer (since 1978/2005) with a real production facility, a notable contract-manufacturing relationship with Parle Products, multi-state distribution, and export reach — strong fit for a manufacturer-focused outreach motion.",
    Hook="Impressive that JRJ Foods/MICHI'S runs a dedicated contract-manufacturing line for Parle Products alongside your own brand — would love to learn how you balance capacity between the two.",
    Sources="Official website (jrjfoods.com, multiple location-specific pages); Tracxn; LinkedIn company page; IndiaMART"
)

# 24 iPaymnt Tech
row(24,
    Segment="Fintech / Banking-as-a-Service (BaaS) Platform",
    WhatTheyMake="B2B open-API fintech platform for banking integration: BBPS bill payments, UPI/Aadhaar Pay, domestic money transfers, mobile recharges, savings/demat account onboarding, and rural neo-banking services",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=4, C1E="Software/API platform business in fintech infrastructure, not a hardware manufacturer",
    C2=10, C2E="Headquartered in New Town, Kolkata, West Bengal, India, with additional offices in Patna and Lucknow",
    C3=7, C3E="Differentiated by managing 250,000+ rural touchpoints and offering an end-to-end white-label fintech-launch ecosystem (regulatory/licensing support, pre-integrated payment gateways, BBPS accreditation assistance) targeting tier-2/tier-3 entrepreneurs",
    C4=NA, C4E="No named CEO/founder/CTO verified despite extensive search",
    C5=8, C5E="Banking-as-a-Service and rural financial inclusion fintech are strong growth areas aligned with India's digital payments push",
    C6=7, C6E="Active PR campaign in June 2025 (ANI press release, published in The Print and The Tribune) announcing a 'ready-to-launch fintech platform' for entrepreneurs — a clear recent growth/marketing signal",
    Verdict="Yes",
    VerdictReasoning="Strong recent media/PR activity (June 2025 national press release), clear fintech infrastructure differentiation (250,000+ rural touchpoints, licensing support), and alignment with high-growth BaaS/financial-inclusion trends, despite lacking a named decision-maker.",
    Hook="Saw iPaymnt Tech's June 2025 launch of a ready-to-launch fintech platform for entrepreneurs — would love to explore how you're scaling licensing/compliance support as adoption grows.",
    Sources="Company website (ipayments.org.in, ipayments.in); ANI press release via The Print (June 2025); The Tribune; Crunchbase; Facebook page; Google Play Store listing"
)

# 25 IRP INNOVATIVE SOLUTION OPC PVT LTD
row(25,
    Segment="IT Services / Software Development",
    WhatTheyMake="Web and mobile application development, ERP solutions, digital marketing services, staff augmentation, and retail/logistics digital solutions",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=3, C1E="Small software development/digital services shop, no manufacturing component",
    C2=10, C2E="Headquartered in Electronic City/Vidyaranyapura, Bangalore, Karnataka, India; active since November 2020",
    C3=4, C3E="Markets specialized capability across retail e-commerce, logistics/supply chain (AI-driven inventory, route optimization), and ERP development, though as a small 7-employee shop",
    C4=NA, C4E="No named founder/CEO verified",
    C5=6, C5E="Web/mobile development and SaaS for retail/logistics digitization continue to see demand in India's SME segment",
    C6=NA, C6E="No funding or major growth news found; remains a small (7-employee) operation",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, active small software development company with positive reviews (4.7/5 on Justdial) and several stated service specializations, but very small scale and no identifiable leadership.",
    Hook=NA,
    Sources="Company website (irpinnovative.com); Facebook page; Justdial; AllIndiaITR; RocketReach"
)

# 26 EntLogics Technologies
row(26,
    Segment="EdTech SaaS / Education Management Software",
    WhatTheyMake="Cloud-native school/education management software suite ('ace') covering admissions, curriculum, billing, document management, and parent/teacher portals for K-12 and coaching centers",
    RevenueBand="$6.2 million (2025, per RocketReach)",
    DecisionMaker=NA,
    DMTitle=NA, DMBackground=NA,
    C1=5, C1E="Cloud-native SaaS product company (education management software) rather than a hardware manufacturer, though it is genuinely a product-engineering business with its own built platform",
    C2=10, C2E="Headquartered in Bangalore, Karnataka, India (founded 2011/2013)",
    C3=6, C3E="Flagship product 'ace' is a comprehensive end-to-end school management suite with claimed partnerships across ~100 schools in India, Malaysia, Hong Kong, and Singapore as of 2014",
    C4=6, C4E="Rajeev Nair identified as Senior Vice President, Global Operations; founding/leadership team described as having 80+ combined years of senior enterprise computing experience, though no current CEO confirmed",
    C5=6, C5E="EdTech/school management SaaS remains a growth category in India and Southeast Asia, though the space is highly competitive (5,200+ active competitors per Tracxn)",
    C6=3, C6E="Sources conflict: Tracxn lists the company as 'not active anymore,' while ZaubaCorp and AllIndiaITR (MCA filings) confirm 'Active' status with a filing as recent as July 2025 — current operating status is genuinely uncertain",
    Verdict="Maybe",
    VerdictReasoning="Genuine product/SaaS company with $6.2M revenue and identifiable senior operations leadership, but conflicting signals on current operating status (Tracxn flags inactive vs. MCA filings showing active) warrant caution before outreach.",
    Hook="Curious whether EntLogics' 'ace' school management suite is still actively expanding given your earlier international footprint across Malaysia, Hong Kong, and Singapore.",
    Sources="ZoomInfo; RocketReach; Tracxn (flags inactive); ZaubaCorp (registry, Active status); AllIndiaITR (Active status, 2025 filing); Justdial"
)

# 27 Infigo Software
row(27,
    Segment="IT Services / Web Design & Digital Marketing",
    WhatTheyMake="Website design and development, e-commerce solutions, digital marketing, SEO, and offshore IT outsourcing services",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=3, C1E="Small web design/IT outsourcing services shop, no manufacturing component",
    C2=10, C2E="Headquartered in Kota, Rajasthan, India (note: distinct from the 'R K Puram' Delhi locality — this R K Puram is a Kota neighborhood)",
    C3=2, C3E="Standard web design/digital marketing offering with no unique IP or certifications found",
    C4=NA, C4E="No named founder/CEO verified",
    C5=5, C5E="Small-business web design/digital marketing demand remains steady in tier-2 Indian cities",
    C6=NA, C6E="No funding or growth news found",
    Verdict="No",
    VerdictReasoning="Small, generic web design/IT outsourcing shop with no identifiable leadership, differentiation, or growth signal.",
    Hook=NA,
    Sources="Company website (infigosoftware.in); LinkedIn company page; Clutch.co; SignalHire; Sulekha; ZoomInfo"
)

# 28 MITESH LODHA
row(28,
    Segment="Fashion / Apparel Retail (Designer Menswear Label)",
    WhatTheyMake="Contemporary Indian menswear (e.g., kurta, sadri sets) under the designer label 'Mitesh Lodha,' including past collections MUJROSA and SAMYIK",
    RevenueBand=NA,
    DecisionMaker="Mitesh Lodha",
    DMTitle="Owner / Founder & Designer",
    DMBackground="Fashion Design graduate from B.D. Somani Institute of Design (2004); came from a family textile business; founded design label Studio Mujrosa, later rebranded; won Lakme Fashion Week Best Fashion Film award (2015)",
    C1=5, C1E="Apparel designer/manufacturer label producing menswear collections, sitting between a small-batch manufacturer and a creative/retail brand",
    C2=10, C2E="Headquartered in Lower Parel, Mumbai, Maharashtra, India",
    C3=6, C3E="Recognized fashion-industry differentiation: Lakme Fashion Week award winner, named contemporary Indian menswear designer with retail presence (High Street Phoenix Mall)",
    C4=3, C4E="Founder/Owner identifiable but is a fashion designer, not a technology decision-maker",
    C5=5, C5E="Indian designer/contemporary menswear retail is a stable niche category",
    C6=NA, C6E="No recent funding, hiring, or expansion news found",
    Verdict="No",
    VerdictReasoning="A small, single-designer fashion label/retail boutique with no technology angle and no scaled manufacturing operation — poor fit for B2B technology or manufacturing-focused outreach despite legitimate brand recognition.",
    Hook=NA,
    Sources="LinkedIn (company and Mitesh Lodha personal profile); Facebook page; Magicpin; Slintel/RocketReach"
)

# 29 Violica
row(29,
    Segment=NA, WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=NA, C1E="Unable to verify",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="No reliable source could confirm the existence or details of a company called 'Violica' at the domain violica.com in Srinagar Colony, Hyderabad; only unrelated businesses with similar names (Viola Skin Clinic, Viola Facility Services, Voila restaurant) were found in the area.",
    Hook=NA,
    Sources="General web search only; no verifiable company profile found"
)

# 30 Heliosys Technologies
row(30,
    Segment="IT Services / Web Design & Development",
    WhatTheyMake="Website design and development (WordPress, HTML/CSS, Shopify/e-commerce), school/college ERP systems, domain/hosting services, and digital marketing (SEO/SMM)",
    RevenueBand=NA,
    DecisionMaker=NA, DMTitle="Owner (no specific name verified beyond a generic LinkedIn 'Owner' profile)",
    DMBackground=NA,
    C1=4, C1E="Web design/IT services and software (school ERP) company, not a hardware manufacturer",
    C2=10, C2E="Headquartered in Indiranagar, Bangalore, Karnataka, India",
    C3=5, C3E="23+ years in business (founded 2002) with 800+ completed projects; differentiated by WCAG/ADA-compliant accessible web design and serving financial services, education, non-profit, and manufacturing sector clients",
    C4=2, C4E="Only a generic, unnamed 'Owner' LinkedIn profile found; no confirmed individual name or title",
    C5=6, C5E="Web design, e-commerce, and accessibility-compliant digital services remain steady demand areas",
    C6=NA, C6E="No specific recent funding or expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Long-tenured (23-year), credible small IT services business with a real project portfolio and accessibility-compliance differentiation, but no identifiable named decision-maker reduces personalization potential.",
    Hook="Heliosys Technologies' 23-year track record building WCAG/ADA-compliant websites for schools and nonprofits stood out — curious how accessibility compliance has shaped your development process.",
    Sources="Company website (heliosystechnologies.com, heliosystechnologies.in); LinkedIn company page; CB Insights; IndiaMART; Facebook page"
)

# 31 Vaishnavi Biotech International Limited
row(31,
    Segment="Biotechnology / Industrial Fermentation Manufacturing",
    WhatTheyMake="Bulk drugs, food preservatives & additives, organic agri-inputs, biofertilizers, and animal health care (cattle/poultry feed supplements) produced via industrial fermentation",
    RevenueBand="$6 million (2025, per RocketReach)",
    DecisionMaker="M. Vaishnavi",
    DMTitle="Head / Founder (referred to as leading 'one of Asia's largest' women-led industrial fermentation establishments)",
    DMBackground="Leads a women- and young-entrepreneur-led team; company is technologically supported by the Prathista R&D Centre of Prathista Industries Limited",
    C1=10, C1E="Genuine industrial fermentation/biotechnology manufacturer producing bulk drugs, food additives, biofertilizers, and animal feed supplements at scale",
    C2=10, C2E="Headquartered in Srikakulam, Andhra Pradesh, India (CSV city 'Sihor' may refer to a specific facility/branch location)",
    C3=8, C3E="Described as one of Asia's largest industrial fermentation establishments led by women; is a 100% Export Oriented Unit; technologically backed by an established R&D center (Prathista Industries) and complies with global certification standards",
    C4=5, C4E="Founder/Head M. Vaishnavi identifiable, though background described in entrepreneurial/leadership terms rather than specifically technical/engineering credentials",
    C5=8, C5E="Industrial biotech, organic/sustainable agri-inputs, and food ingredient manufacturing are strong growth categories, especially for export-oriented Indian manufacturers",
    C6=6, C6E="$6M revenue with export-oriented operations and active hiring per ZoomInfo, though no specific recent funding round identified",
    Verdict="Yes",
    VerdictReasoning="Genuine, scaled industrial biotech manufacturer with strong export orientation, notable women-led leadership, R&D backing, and global certification compliance — a strong fit for manufacturer-focused outreach.",
    Hook="Your team's work as one of Asia's largest women-led industrial fermentation manufacturers, backed by Prathista's R&D center, is a compelling story — would love to learn how you're scaling export compliance across your bio-input lines.",
    Sources="RocketReach; Crunchbase; ZoomInfo"
)

# 32 Stratrun
row(32,
    Segment="Professional Services / Freelance Project Marketplace",
    WhatTheyMake="On-demand business services matching (skill-set matching for projects) rather than a defined product",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Project/freelance services-matching platform, no manufacturing component",
    C2=NA, C2E="Domain is stratrun.in suggesting an Indian operation, but no specific city/registry record found beyond source data's 'Delhi'",
    C3=NA, C3E="Unable to verify specific differentiation beyond the company's own generic 'how it works' page",
    C4=NA, C4E="No named founder/CEO verified",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="Only the company's own website could be found describing a generic freelance-matching service; no independent verification (LinkedIn, Crunchbase, Tracxn, news) exists to confirm scale, leadership, or legitimacy.",
    Hook=NA,
    Sources="Company website (stratrun.in) only; no independent verification found"
)

# 33 Dhanrashi Fintech
row(33,
    Segment="Fintech / Microfinance (SME & Micro Loans)",
    WhatTheyMake="Diversified SME and micro loan services positioned as a hybrid Fin-Tech and Microfinance Institution (MFI), offering paperless lending solutions",
    RevenueBand=NA,
    DecisionMaker="Amit Kumar",
    DMTitle="Co-Founder, CEO & Director",
    DMBackground="Co-founded Dhanrashi in 2019/2021 alongside Puneet Bansal (Co-Founder & Director, since departed per company filings) and Prabhat Agrawal (Director)",
    C1=3, C1E="Fintech/MFI lending services business, no manufacturing component",
    C2=10, C2E="Headquartered in Gurugram, Haryana, India",
    C3=4, C3E="Positions itself as a 'hybrid Fin-Tech and MFI' offering technology-driven, paperless micro and SME lending, though faces 679 active competitors per Tracxn including major players like L&T Finance and Manappuram Finance",
    C4=6, C4E="Co-Founder & CEO Amit Kumar clearly identified via Tracxn and MCA filings (also a registered Director)",
    C5=6, C5E="SME/micro-lending and financial inclusion remain growth areas in India's fintech sector",
    C6=2, C6E="Tracxn reports a dramatic 95% decline in employee count between May 2024 and May 2025 (from roughly 100 down to 5 employees), a significant negative growth signal despite the company remaining formally 'Active' per MCA filings",
    Verdict="No",
    VerdictReasoning="Despite a clearly identified CEO and active legal status, the company shows a severe (95%) recent employee headcount decline, suggesting significant business contraction — a strong negative signal against outreach prioritization.",
    Hook=NA,
    Sources="Tracxn (employee count trend); LinkedIn company page; ZaubaCorp; Tofler; IndiaFilings; TheCompanyCheck (MSME registration)"
)

# 34 Finance Bridge
row(34,
    Segment="Financial Services (Commercial Finance, Consumer Loans & Wealth Services)",
    WhatTheyMake="Commercial finance, consumer loan products, and wealth management/distribution services",
    RevenueBand=NA,
    DecisionMaker="Vaibhav Sharma",
    DMTitle="Finance Executive (specific senior leadership/founder not separately confirmed)",
    DMBackground=NA,
    C1=2, C1E="Pure financial services firm, no manufacturing component",
    C2=9, C2E="Registered with the Government of Tamil Nadu (MRCT) and recognized by the Ministry of Micro, Small and Medium Enterprises (MSME) and Central Board for Direct Taxes (CBDT), indicating an India-based, Tamil Nadu-anchored operation",
    C3=3, C3E="Standard commercial finance/consumer loan/wealth distribution offering with formal government registrations (MRCT, MSME, CBDT) but no unique product IP found",
    C4=2, C4E="Only a Finance Executive (Vaibhav Sharma) identified — no confirmed senior/tech decision-maker",
    C5=6, C5E="Commercial finance and consumer lending continue to see steady demand in India",
    C6=NA, C6E="No specific recent funding, hiring, or expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Formally registered financial services firm with multiple government recognitions, but no senior decision-maker identified and limited differentiation.",
    Hook=NA,
    Sources="RocketReach"
)

# 35 Galaxykart
row(35,
    Segment=NA, WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=NA, C1E="Unable to verify",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="No reliable source could confirm the existence or details of a company at the domain galaxykart.in in R K Puram; search results returned only an unrelated VR racing game ('Galaxy Kart VR') and general Indian e-commerce sector information, confirming the source data's 'Website unreachable or error' finding.",
    Hook=NA,
    Sources="General web search only; no verifiable company profile found"
)

# 36 CMFA
row(36,
    Segment="Financial Services / Business & Financial Advisory",
    WhatTheyMake="Financial management consulting, loan and subsidy application assistance, project report preparation, and job-oriented skills training for entrepreneurs and low-to-moderate income individuals",
    RevenueBand="<$5 million (per ZoomInfo)",
    DecisionMaker="James George",
    DMTitle="Director",
    DMBackground="Over two decades of experience in management and finance; leads CMFA's mission to support entrepreneurial development, self-employment schemes, and government loan/subsidy access for underserved individuals",
    C1=2, C1E="Pure financial advisory and training services business, no manufacturing component",
    C2=10, C2E="Headquartered in Kochi, Kerala, India",
    C3=4, C3E="Differentiated by its social-impact mission serving low-to-moderate income individuals, women entrepreneurs, and small businesses with loan/subsidy navigation and job-oriented courses",
    C4=3, C4E="Director James George identifiable, but background is in finance/management rather than technology",
    C5=5, C5E="MSME loan/subsidy advisory and entrepreneurship support continue to see steady demand in Kerala's small business ecosystem",
    C6=3, C6E="Small team (1-10 employees) with no specific recent funding, hiring, or expansion news found beyond routine social media activity",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, small, social-impact-oriented financial advisory firm with an identified director, but minimal scale and no technology angle.",
    Hook=NA,
    Sources="Company website (cmfakochi.com, Our Story page); LinkedIn company page; Facebook page; ZoomInfo; Yellow Pages"
)

# 37 EnglishWithShirley
row(37,
    Segment="Education / Language Training (Individual Practitioner)",
    WhatTheyMake="English language training including IELTS preparation, spoken/business/banking English, and CBSE school-curriculum English tutoring",
    RevenueBand=NA,
    DecisionMaker="Shirley Denis",
    DMTitle="Founder & Lead Instructor",
    DMBackground="Started EnglishWithShirley in 2000; has personally taught approximately 1,000+ students over more than two decades across IELTS, business English, and school curriculum levels",
    C1=1, C1E="Individual-led English language tutoring/training practice, no manufacturing or product component",
    C2=10, C2E="Headquartered in Lajpat Nagar, New Delhi, India, with apparent expansion content also targeting Mumbai",
    C3=5, C3E="24+ years of teaching experience differentiates the practice, with structured offerings across IELTS, business/banking English, and school-curriculum levels",
    C4=1, C4E="Founder is an English language educator, not a technology decision-maker",
    C5=4, C5E="English-language and IELTS coaching demand remains steady in India, particularly for study-abroad and career-advancement seekers",
    C6=3, C6E="Active recent content marketing (2026-dated blog posts targeting Mumbai market) suggests ongoing/expanding operations, though no funding or major scale signals",
    Verdict="No",
    VerdictReasoning="A small, individual-led language tutoring practice with no technology, manufacturing, or scaled-business characteristics — poor fit for B2B technology/manufacturing-focused outreach despite legitimate, long-standing operations.",
    Hook=NA,
    Sources="LinkedIn company page; Company website (englishwithshirley.com); UrbanPro listing"
)

# 38 ADVAIT BUSINESS SOLUTIONS PRIVATE LIMITED
row(38,
    Segment="IT Services / SAP Consulting",
    WhatTheyMake="SAP software implementation, upgrades, migrations, and ongoing support; custom web-based software development; SAP S/4 HANA rollouts and cloud migration services",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=4, C1E="SAP/IT consulting services company serving enterprise clients, not a hardware/physical manufacturer",
    C2=10, C2E="Headquartered in Prahladnagar, Ahmedabad, Gujarat, India, with additional offices in Chennai and Hyderabad",
    C3=5, C3E="Established SAP Partner status with services spanning ERP, Financials, HCM, SCM, and AI/ML-enabled SAP analytics; positions itself with active thought-leadership content on SAP AI/ML capabilities",
    C4=NA, C4E="No named founder/CEO verified despite multiple sources; named staff found are mid-level (Business Development Executive, Regional Manager, SAP FI Consultant)",
    C5=7, C5E="SAP consulting and enterprise digital transformation continue to see strong demand as Indian businesses modernize ERP systems",
    C6=4, C6E="10-50 employees (SignalHire) with active LinkedIn content marketing, but no specific funding or major expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, multi-city SAP consulting partner with active marketing and a reasonable employee base, but no identifiable senior/founder-level decision-maker reduces personalization confidence.",
    Hook="ADVAIT's recent content on SAP's AI/ML-driven analytics capabilities suggests you're pushing clients toward predictive SAP use cases — curious how that's landing with your enterprise customers.",
    Sources="LinkedIn company page; Crunchbase; SAP Partner Finder; ContactOut; SignalHire; SalesZshark; Facebook pages"
)

# 39 Lakewater Advisors
row(39,
    Segment="Financial Services / SEBI-Registered Portfolio Management Services (PMS)",
    WhatTheyMake="Portfolio Management Services (PMS) focused on concentrated, long-term investments in high-quality publicly listed Indian equities; also offers private equity/unlisted shares advisory",
    RevenueBand=NA,
    DecisionMaker="Pankaj Singhania",
    DMTitle="Co-Founder & Fund Manager (also listed as Founder and Director)",
    DMBackground="Chartered Accountant (CA), Cost Accountant, and former Civil Servant; over 25 years of experience in capital and financial markets, policy-making, and corporate governance",
    C1=2, C1E="Pure financial services/asset management firm, no manufacturing component",
    C2=10, C2E="Headquartered in Salt Lake, Sector V, Kolkata, West Bengal, India, with an additional office in Bandra Kurla Complex, Mumbai",
    C3=7, C3E="SEBI-registered PMS provider (Registration No. INP000006767) with a distinctive long-term, governance-focused 'Conscious Corporates' investment philosophy; publishes regular equity research notes on companies like Adani Energy Solutions and Hindustan Aeronautics",
    C4=5, C4E="Co-Founder & Fund Manager Pankaj Singhania clearly identified, with strong financial/regulatory credentials, though not a technology-focused decision-maker",
    C5=7, C5E="Portfolio Management Services and India-focused equity investment continue to see strong tailwinds from rising domestic wealth and India's growth narrative",
    C6=6, C6E="Reported a 20.6% revenue CAGR since FY2022 per the company's own site; small team (4 employees per Tracxn) but consistent SEBI top-performer rankings claimed",
    Verdict="Yes",
    VerdictReasoning="SEBI-regulated, credible PMS firm with a clearly identified, highly credentialed founder (25+ years experience), genuine differentiation through its governance-focused investment philosophy, and a reported 20.6% revenue CAGR — a solid outreach candidate.",
    Hook="Lakewater's 20.6% revenue CAGR since FY2022 alongside your SEBI top-performer PMS rankings is a strong growth story — would love to learn how your team scales equity research as AUM grows.",
    Sources="Lakewater Advisors official website (lakewateradvisors.com, About page); PMS Bazaar; Tracxn; Sharescart; LinkedIn company page; Facebook page"
)

# 40 CipherTronIndia
row(40,
    Segment="IT Services / Web Design",
    WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=3, C1E="Appears to be a small web design/IT services business based on limited 'About Venor' page content found at the domain, though this could not be fully confirmed",
    C2=NA, C2E="Domain suggests an Indian operation ('India' in name) but no registry record or city confirmed beyond source data's 'Delhi'",
    C3=NA, C3E="Unable to verify specific differentiation",
    C4=NA, C4E="No named founder/CEO verified",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="Only a fragmentary 'About Us' page snippet (referencing an unrelated business name 'Venor') was found at ciphertron.in; no LinkedIn, Crunchbase, Tracxn, or news profile could confirm this as a legitimate, currently operating, identifiable company.",
    Hook=NA,
    Sources="Partial company website snippet only (ciphertron.in/about-us); no independent verification found"
)

# 41 Early Wages
row(41,
    Segment="Fintech / Earned Wage Access (EWA)",
    WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=3, C1E="If operational, would be an Earned Wage Access fintech platform (software/API service), not a manufacturer",
    C2=NA, C2E="Domain (earlywages.in) suggests an Indian operation but no registry record, city, or independent verification found",
    C3=NA, C3E="Unable to verify; numerous other named EWA players exist in India (Jify, PayWings, Emerald Finance, Refyne) but none confirmed to be this specific company",
    C4=NA, C4E="Unable to verify",
    C5=7, C5E="Earned Wage Access is a genuinely fast-growing fintech category in India given rising employer/employee adoption of on-demand pay",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="Only a bare landing page was found at earlywages.in with no further verifiable content; this is consistent with the source data's 'Website unreachable or error' finding, and no independent source (LinkedIn, Crunchbase, news) could confirm the company's specific identity, leadership, or scale.",
    Hook=NA,
    Sources="Bare website found (earlywages.in); no independent verification located"
)

# 42 Lumynos Technologies
row(42,
    Segment="IT Services / Odoo ERP Implementation",
    WhatTheyMake="Odoo ERP and e-commerce implementation, customization, integration, and migration services, including custom Odoo module development",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=4, C1E="ERP/e-commerce software implementation services company, not a hardware manufacturer",
    C2=10, C2E="Headquartered in Gota, Sabarmati, Gujarat, India",
    C3=5, C3E="Specialized niche focus specifically on Odoo ERP (rather than generalist ERP consulting), with published Odoo Apps Store modules under the company name",
    C4=NA, C4E="No named founder/CEO verified",
    C5=6, C5E="Odoo and open-source ERP adoption continue to grow among SMEs seeking cost-effective ERP alternatives",
    C6=NA, C6E="No specific funding or major growth news found",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, niche-focused Odoo ERP implementation partner with verifiable published work (Odoo Apps Store listings), but no identifiable leadership reduces personalization potential.",
    Hook=NA,
    Sources="Company website (lumynostechnologies.com); ZoomInfo; Odoo Apps Store (published modules); RemoteHub"
)

# 43 Deaara
row(43,
    Segment="Creative Services / Branding & Digital Marketing Agency",
    WhatTheyMake="Brand strategy, logo/packaging/stationery design, website and mobile app design/development, and social media/digital marketing services",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Creative/branding and digital design agency, no manufacturing component",
    C2=10, C2E="Headquartered in Vastrapur, Ahmedabad, Gujarat, India",
    C3=5, C3E="16+ years in business (founded 2009) as an award-winning brand and digital consultancy, with named client case studies (Vanilla bakery brand, Navkar Trading Co. kitchen appliance manufacturer, a real estate company)",
    C4=NA, C4E="No named founder/CEO verified",
    C5=5, C5E="Branding and digital marketing services demand remains steady among Indian SMEs",
    C6=NA, C6E="No specific recent funding or expansion news found",
    Verdict="No",
    VerdictReasoning="Established, credible branding/creative agency with real client case studies, but pure-services model with no technology or manufacturing angle and no identifiable senior decision-maker.",
    Hook=NA,
    Sources="Company website (deaara.com, About Us, capabilities pages); Facebook page; Justdial"
)

# 44 Iamo Bazaar
row(44,
    Segment="Retail-Tech / Cashback & Loyalty Platform (DEFUNCT)",
    WhatTheyMake="App-based cashback platform offering multi-category cashback (fashion, beauty, home decor, electronics) on purchases at partnered local/offline stores",
    RevenueBand=NA,
    DecisionMaker="Manas Kumar Sahoo",
    DMTitle="Founder & CEO",
    DMBackground="Founded IAMO Solutions Pvt Ltd (project: IAMO Bazaar) in 2017; company employed a multi-level referral/commission structure for both consumers and partner stores",
    C1=2, C1E="Cashback/loyalty app platform business, no manufacturing component",
    C2=10, C2E="Headquartered in Rasulgarh, Bhubaneswar, Odisha, India",
    C3=3, C3E="Multi-level cashback referral model spanning fashion, beauty, home decor/improvement, and electronics categories, competing against established players like magicpin and NearBuy",
    C4=5, C4E="Founder & CEO Manas Kumar Sahoo identifiable",
    C5=4, C5E="Offline-to-online cashback/loyalty apps face strong competition from larger, better-funded players (magicpin, NearBuy)",
    C6=0, C6E="Tracxn explicitly classifies IAMO Bazaar as 'deadpooled' (defunct); the company's referral/commission income structure also resembles characteristics of multi-level marketing (MLM) schemes per independent Hindi-language explainer content",
    Verdict="No",
    VerdictReasoning="Independent source (Tracxn) confirms the company is deadpooled/defunct, and its underlying referral-income business model shows MLM-like characteristics — not a viable or advisable outreach target.",
    Hook=NA,
    Sources="Tracxn (deadpooled status); RocketReach (Iamo Bazaar Management); Sulekha; Justdial; Facebook pages; jankarinew.com (Hindi business-model explainer)"
)

# 45 Quick Serve Financial Services
row(45,
    Segment="Financial Services / Loan Brokerage",
    WhatTheyMake="MSME business loans (secured and unsecured), home loans, and loan-against-property products sourced through banking/NBFC partnerships",
    RevenueBand=NA,
    DecisionMaker="Shadab Khan",
    DMTitle="Founder",
    DMBackground="14+ years of financial services experience in Asset Finance prior to founding Quick Serve Financial in 2016; previously worked at Standard Chartered Bank, ABN AMRO Bank, and Deutsche Bank",
    C1=2, C1E="Pure loan brokerage/financial consulting business, no manufacturing component",
    C2=10, C2E="Headquartered in Mangalwar Peth, Pune, Maharashtra, India",
    C3=4, C3E="Differentiated mainly by founder's prior banking pedigree (Standard Chartered, ABN AMRO, Deutsche Bank) rather than unique product IP; offers multiple MSME loan schemes",
    C4=4, C4E="Founder Shadab Khan identifiable, with a strong banking background, though not a technology decision-maker",
    C5=6, C5E="MSME loan demand remains a consistent growth driver in India given the formal credit gap",
    C6=4, C6E="Active LinkedIn posting (2-10 employees) showing ongoing MSME loan and home loan marketing campaigns; no major funding or expansion news found",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, founder-led loan brokerage with strong banking pedigree, but small scale, pure-services model, and no technology differentiation.",
    Hook=NA,
    Sources="Company website (quickservefinancial.com, mission-vision page); LinkedIn company page; Instagram; Justdial; Sulekha"
)

# 46 Sauramahi
row(46,
    Segment=NA, WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=NA, C1E="Unable to verify",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="No reliable source could confirm the existence or details of a company called 'Sauramahi' / 'Saurmahi' at the domain saurmahi.com in Pune; search results returned only unrelated fish/seafood terminology content and an unrelated company ('Sauram Foods India Pvt Ltd'), confirming the source data's 'Website unreachable or error' finding.",
    Hook=NA,
    Sources="General web search only; no verifiable company profile found"
)

# 47 Inside View Technologies India Private Limited
row(47,
    Segment="B2B SaaS / Sales & Marketing Intelligence Platform",
    WhatTheyMake="AI-powered Targeting Intelligence platform providing B2B company/contact data, go-to-market decision support, and sales/marketing alignment tools (parent: InsideView, now part of the Demandbase/ZoomInfo ecosystem)",
    RevenueBand="$25 million (2026, per RocketReach)",
    DecisionMaker="Sesha Rao",
    DMTitle="Board Member, InsideView India",
    DMBackground=NA,
    C1=5, C1E="Genuine AI-powered B2B SaaS data platform/product company (Targeting Intelligence), though software rather than physical manufacturing",
    C2=8, C2E="India subsidiary (Inside View Technologies India Private Limited) headquartered in Hyderabad, Telangana, India, with 71 employees locally; parent company InsideView is US-headquartered",
    C3=7, C3E="Established AI-driven B2B sales intelligence platform with recognized brand (InsideView) trusted by 'the world's leading B2B companies' per company messaging; differentiated by combining firmographic data with AI-driven targeting",
    C4=6, C4E="Board Member Sesha Rao identified for the India entity, though specific CTO/India country-head title not separately confirmed",
    C5=8, C5E="B2B sales intelligence, data-driven GTM, and AI-powered targeting are strong, growing categories in enterprise software",
    C6=4, C6E="$25M revenue and 71 employees reported for the India subsidiary, but no specific recent India-level funding or expansion news found (parent company has undergone industry consolidation in this space)",
    Verdict="Yes",
    VerdictReasoning="Substantial, well-established AI-powered B2B SaaS platform with meaningful India operations ($25M revenue, 71 employees) and a recognized global brand — a credible, scaled technology company for outreach.",
    Hook="InsideView's AI-driven Targeting Intelligence platform has a strong reputation with leading B2B companies globally — would love to explore how your Hyderabad team supports platform delivery at scale.",
    Sources="RocketReach; LinkedIn company page; RocketReach Email Format listing"
)

# 48 Savor Experiences
row(48,
    Segment="Hospitality / Events & Culinary Experiences",
    WhatTheyMake="Curated food events, supper clubs, and gourmet lunch experiences",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Events/hospitality services business, no manufacturing component",
    C2=10, C2E="Headquartered in Lower Parel, Mumbai, Maharashtra, India",
    C3=4, C3E="13+ years in business (founded 2012) with a specific niche focus on supper clubs and gourmet lunch events, differentiating from standard catering",
    C4=NA, C4E="No named founder/CEO verified",
    C5=4, C5E="Curated food/events experiences are a steady but niche segment within India's hospitality sector",
    C6=NA, C6E="No specific recent funding, hiring, or expansion news found",
    Verdict="No",
    VerdictReasoning="Small (2-10 employee), niche events/hospitality business with no technology, manufacturing, or scaled-growth signal, and no identifiable decision-maker.",
    Hook=NA,
    Sources="LinkedIn company page; Facebook page"
)

# 49 Hibilter IT Systems Pvt Ltd
row(49,
    Segment="Enterprise Software / AI & Product Development",
    WhatTheyMake="Enterprise AI platform (HAOW) offering air-gapped/on-premise generative AI, document intelligence, knowledge assistants, computer vision, and autonomous AI agents; also full-spectrum software product development services",
    RevenueBand=NA,
    DecisionMaker="Suresh Tiwari",
    DMTitle="CEO",
    DMBackground=NA,
    C1=6, C1E="Genuine enterprise AI/software product company (HAOW platform) with deployable infrastructure across on-premise, private cloud, and edge devices, positioning it closer to a product/technology manufacturer than a pure services shop",
    C2=10, C2E="Headquartered in New Delhi, India (Krishna Nagar, South West Delhi), with an Offshore Development Center also in New Delhi",
    C3=8, C3E="Strong differentiation: HAOW platform offers complete data sovereignty, true air-gapped AI with zero external dependencies, and OpenAI-compatible APIs without vendor lock-in — serving Defense & National Security, Government & Public Sector, and Healthcare & Life Sciences clients",
    C4=8, C4E="CEO Suresh Tiwari clearly identified via RocketReach; company also has a separate authorized representative (Ankesh Tiwari) listed on Refrens",
    C5=9, C5E="Air-gapped/sovereign enterprise AI for defense, government, and healthcare is a high-growth, high-relevance category amid rising data sovereignty and AI security concerns",
    C6=5, C6E="9+ years in business (founded 2016) with a notable pivot toward AI/ML, deep learning, and blockchain over the past 3 years per Refrens; employee count estimates vary widely (4 per RocketReach vs. 100-200 per SignalHire)",
    Verdict="Yes",
    VerdictReasoning="Genuinely differentiated, defense/government/healthcare-focused sovereign AI product company with a clearly identified CEO and a compelling air-gapped, vendor-lock-in-free AI platform — strong outreach candidate, particularly for security-conscious enterprise AI conversations.",
    Hook="Hibilter's air-gapped HAOW platform for Defense and Healthcare clients is a timely story given rising data sovereignty demands — would love to learn how you're scaling secure AI deployment across those sectors.",
    Sources="RocketReach; TheOrg; Refrens; Company website (hibilter.com); ZaubaCorp; AllIndiaITR; SignalHire"
)

# 50 Algonetix Technologies
row(50,
    Segment="Digital Marketing / SEO Agency",
    WhatTheyMake="SEO, PPC, social media marketing/optimization, website design and development, and local business optimization services",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=2, C1E="Pure digital marketing/SEO agency, no manufacturing component",
    C2=10, C2E="Headquartered in Dwarka, New Delhi, India; founded 2021",
    C3=3, C3E="Standard full-service digital marketing offering (SEO, PPC, SMM, web design) with positive customer reviews but no unique IP found",
    C4=NA, C4E="No named founder/CEO verified",
    C5=5, C5E="SEO/digital marketing demand remains steady among Indian SMEs seeking online visibility",
    C6=3, C6E="Positive Trustindex-verified Google reviews and active blog/content marketing suggest steady operations, though no funding or major growth news found",
    Verdict="No",
    VerdictReasoning="Small (2-10 employee), generic digital marketing agency with no technology/manufacturing angle and no identifiable senior decision-maker.",
    Hook=NA,
    Sources="Company website (algonetix.in); LinkedIn company page; about.me; Clutch.co; LeadStal directory"
)

# 51 G K Vale & Co.
row(51,
    Segment="Retail / Photography Products & Services",
    WhatTheyMake="Professional and consumer photography services (wedding, portrait, passport/visa photos), photo printing products (canvas prints, photo books, frames), and Canon camera retail (exclusive Canon Image Square stores)",
    RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=5, C1E="Operates its own state-of-the-art production facility for photo printing/products (per company messaging) alongside retail and services, blending light manufacturing with retail/services",
    C2=10, C2E="Headquartered in Fraser Town, Bangalore, Karnataka, India",
    C3=7, C3E="115-year operating history (since 1910) with 35+ photo stores, a dozen exclusive Canon Image Square stores, and 3 multi-brand camera stores across Karnataka; strong heritage brand differentiation with an established e-commerce portal since 2016",
    C4=NA, C4E="No named CEO/founder/decision-maker verified",
    C5=5, C5E="Traditional photography retail/printing faces ongoing disruption from smartphone photography and digital sharing, though demand persists for weddings, portraits, and official document photos",
    C6=4, C6E="Active hiring (Portrait Photographer roles) and recent new store openings (Bangalore North - Bagalur Cross) indicate ongoing physical expansion, though no major funding or digital transformation news found",
    Verdict="Maybe",
    VerdictReasoning="Legitimate, long-established heritage retail/photo-products business with real production facilities and ongoing physical expansion, but no identifiable senior decision-maker and a legacy retail model facing structural industry headwinds.",
    Hook="115 years in photography and still opening new stores (like your recent Bagalur Cross location) is a remarkable legacy — curious how G K Vale balances heritage retail with your e-commerce expansion since 2016.",
    Sources="LinkedIn company page; ZoomInfo; Justdial (multiple store listings); X/Twitter; YouTube channel; WedMeGood"
)

# 52 TEBS TECH
row(52,
    Segment=NA, WhatTheyMake=NA, RevenueBand=NA, DecisionMaker=NA, DMTitle=NA, DMBackground=NA,
    C1=NA, C1E="Unable to verify",
    C2=NA, C2E="Unable to verify",
    C3=NA, C3E="Unable to verify",
    C4=NA, C4E="Unable to verify",
    C5=NA, C5E="Unable to verify",
    C6=NA, C6E="Unable to verify",
    Verdict="No",
    VerdictReasoning="The domain tebstech.com at Aarey Milk Colony, Goregaon (Mumbai) could not be independently verified. Notably, an unrelated jobs-fraud-awareness directory lists 'Tabstech Technologies Pvt Ltd / Tebstech Technologies Pvt Ltd' as a suspected/fake company name pattern used in fraudulent job postings, which raises caution; a separate, unrelated logistics-optimization company also uses the similar name 'TEBS' (tebs.co.in) but does not match this domain or location. Given the inability to verify legitimacy and the fraud-pattern flag, this entity should not be pursued without further independent confirmation.",
    Hook=NA,
    Sources="techprozetta.blogspot.com (suspected/fake companies list, flags similar name pattern); general web search; no independent verification of legitimacy found"
)

# 53 Payism Global
row(53,
    Segment="Fintech / Agent-Based Financial Services Network",
    WhatTheyMake="Domestic money remittances, telecom recharge, bill payment (Super POS, Micro ATM), airline/bus/hotel/railway ticketing, assisted e-commerce, agritech, microfinance, and insurance distribution via an agent/retailer network",
    RevenueBand="₹136 Cr (FY ending Mar 2025, per Tracxn) / $31.7 million (per ZoomInfo)",
    DecisionMaker="Uday Kumar (CEO) / Nandish Domlur (Founder & MD)",
    DMTitle="Founder & CEO / Founder & Managing Director",
    DMBackground="Nandish Domlur and Uday Kumar co-founded Payism in 2013/2014, building it into a B2B2C agent-network fintech platform; backed by two high-net-worth individuals from Singapore and Bangalore; reported reaching ₹5,000 Cr platform turnover at the 3-year mark per a SiliconIndia interview",
    C1=4, C1E="Agent-network fintech platform/software business facilitating financial transactions through a distributed retailer network, not a physical manufacturer",
    C2=10, C2E="Headquartered in Domlur, Bangalore, Karnataka, India (ZoomInfo separately notes a Madhapur, Telangana office)",
    C3=6, C3E="Differentiated B2B2C agent-network model serving 1,300+ distributors and 28,000+ retailers, explicitly targeting domestic migrants and the unbanked population across remittances, ticketing, agritech, and microfinance",
    C4=8, C4E="Both Founder & CEO (Uday Kumar) and Founder & Managing Director (Nandish Domlur) clearly and consistently identified across multiple sources (Tracxn, Owler, EasyLeadz, ZoomInfo)",
    C5=7, C5E="Agent-based financial inclusion services for migrant and unbanked populations remain a strong tailwind in India's fintech/BaaS landscape",
    C6=5, C6E="Reported ₹136 Cr annual revenue with 106 employees (up 16% YoY as of July 2024) per Tracxn, though Tracxn separately classifies the company as 'deadpooled' (potentially conflicting with the positive revenue/headcount trend, indicating uncertain current status)",
    Verdict="Maybe",
    VerdictReasoning="Strong historical scale (₹136 Cr revenue, 28,000+ retailer network) and clearly identified founders, but a conflicting 'deadpooled' classification from Tracxn alongside otherwise positive growth metrics creates real uncertainty about current operating status that should be confirmed before outreach.",
    Hook="Payism's agent network of 28,000+ retailers serving India's unbanked population is an impressive distribution model — would love to confirm how the platform is evolving and explore where we could add value.",
    Sources="Tracxn (revenue, employee trend, deadpooled status flag); LinkedIn company page; Owler; EasyLeadz; ZoomInfo; SiliconIndia (founder interview); Internshala"
)

# Read the original CSV
with open("/content/final_submission.csv", "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    fieldnames = reader.fieldnames

# Update rows
for idx, r in enumerate(rows):
    d = data.get(idx, {})

    r['Segment'] = d.get('Segment', NA)
    r['What They Make'] = d.get('WhatTheyMake', NA)
    r['Revenue Band'] = d.get('RevenueBand', NA)

    if d.get('DecisionMaker'):
        r['Decision Maker'] = d['DecisionMaker']

    if d.get('DMTitle'):
        r['DM Title'] = d['DMTitle']

    r['DM Background'] = d.get('DMBackground', NA)

    r['C1 Manufacturer Score'] = d.get('C1', NA)
    r['C1 Evidence'] = d.get('C1E', NA)
    r['C2 India Score'] = d.get('C2', NA)
    r['C2 Evidence'] = d.get('C2E', NA)
    r['C3 Differentiation Score'] = d.get('C3', NA)
    r['C3 Evidence'] = d.get('C3E', NA)
    r['C4 Tech DM Score'] = d.get('C4', NA)
    r['C4 Evidence'] = d.get('C4E', NA)
    r['C5 Tailwind Score'] = d.get('C5', NA)
    r['C5 Evidence'] = d.get('C5E', NA)
    r['C6 Growth Score'] = d.get('C6', NA)
    r['C6 Evidence'] = d.get('C6E', NA)

    scores = [d.get('C1'), d.get('C2'), d.get('C3'),
              d.get('C4'), d.get('C5'), d.get('C6')]

    if all(isinstance(s, (int, float)) for s in scores):
        fed = sum(scores)
        r['Federer Score'] = fed

        if fed <= 20:
            r['Score Band'] = "Low"
        elif fed <= 35:
            r['Score Band'] = "Medium"
        elif fed <= 50:
            r['Score Band'] = "High"
        else:
            r['Score Band'] = "Excellent"
    else:
        r['Federer Score'] = NA
        r['Score Band'] = NA

    r['Verdict'] = d.get('Verdict', NA)
    r['Verdict Reasoning'] = d.get('VerdictReasoning', NA)
    r['Personalization Hook'] = d.get('Hook', NA)
    r['Evidence Sources'] = d.get('Sources', NA)

# Write the completed CSV
with open("completed_submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"✅ completed_submission.csv has been created with {len(rows)} rows.")

✅ completed_submission.csv has been created with 54 rows.


In [35]:
import pandas as pd

# Load the completed_submission.csv file
df_completed_submission = pd.read_csv('/content/completed_submission.csv')

# Filter out rows where 'Segment' is null
df_completed_submission = df_completed_submission[df_completed_submission['Segment'].notna()]

# Calculate the number of non-null values for each row across all columns
df_completed_submission['non_null_count'] = df_completed_submission.count(axis=1)

# Sort the DataFrame by 'non_null_count' in descending order
df_sorted_by_completeness = df_completed_submission.sort_values(by='non_null_count', ascending=False)

# Select the top 25 companies
top_25_companies = df_sorted_by_completeness.head(25)

# Reset the index to get sequential serial numbers
top_25_companies = top_25_companies.reset_index(drop=True)

print('Top 25 companies based on data completeness (fewer null/NA values) with non-null Segments:')
display(top_25_companies)

Top 25 companies based on data completeness (fewer null/NA values) with non-null Segments:


,Company Name,Website,City,Segment,What They Make,Revenue Band,Decision Maker,DM Title,DM Background,C1 Manufacturer Score,...,C5 Evidence,C6 Growth Score,C6 Evidence,Federer Score,Score Band,Verdict,Verdict Reasoning,Personalization Hook,Evidence Sources,non_null_count
0,Kashmir Box,kashmirbox.com,Srinagar,E-commerce / Handicrafts & Artisanal Products ...,"Online marketplace for Kashmiri handicrafts, h...","₹5.62 Cr (FY2024, per Tracxn)",Muheet Mehraj,Co-Founder & CEO,Co-founded Kashmir Box in 2011 with Kashif Ahm...,6.0,...,Global handicrafts/artisanal e-commerce and so...,8.0,Raised a Seed round in October 2025 (total $23...,47.0,High,Yes,"Well-documented, actively funded company (late...",Congrats on closing your Seed round in October...,Tracxn; Owler; YourStory (2015 founder intervi...,27
1,TivonaGlobal Technologies,tivonaglobal.com,Alwarpet,IT Services / Cloud & DevOps Consulting,"Cloud computing, DevOps automation, infrastruc...","$6 million (2026, per RocketReach)",Dharma Prakash,Co-Founder & Chief Executive Officer,Co-founded TivonaGlobal in mid-2017 alongside ...,4.0,...,"Cloud migration, DevOps, and infrastructure mo...",5.0,60 employees and $6M revenue reported (RocketR...,42.0,High,Yes,Clearly identified technical co-founders (CEO ...,Noticed TivonaGlobal is a certified Google Clo...,LinkedIn company page; ZoomInfo; RocketReach; ...,27
2,Vaishnavi Biotech International Limited,vaishnavibiotech.com,Sihor,Biotechnology / Industrial Fermentation Manufa...,"Bulk drugs, food preservatives & additives, or...","$6 million (2025, per RocketReach)",M. Vaishnavi,Head / Founder (referred to as leading 'one of...,Leads a women- and young-entrepreneur-led team...,10.0,...,"Industrial biotech, organic/sustainable agri-i...",6.0,$6M revenue with export-oriented operations an...,47.0,High,Yes,"Genuine, scaled industrial biotech manufacture...",Your team's work as one of Asia's largest wome...,RocketReach; Crunchbase; ZoomInfo,27
3,MICHI'S,jrjfoods.com,Manual Review,Manufacturing / FMCG Food & Confectionery,"Candies, toffees, and toffee bars (brand: MICH...","₹18.6 Cr (FY ending Mar 2025, per Tracxn)",Jagdish Thakkar,Co-Founder & Managing Director (also Joint Man...,"Born 1949, grew the company from its founding ...",10.0,...,"FMCG confectionery is a stable, growing catego...",6.0,₹18.6 Cr annual revenue with 68 employees (sli...,44.0,High,Yes,"Genuine, well-established FMCG manufacturer (s...",Impressive that JRJ Foods/MICHI'S runs a dedic...,"Official website (jrjfoods.com, multiple locat...",27
4,Payism Global,payismglobal.com,Bangalore,Fintech / Agent-Based Financial Services Network,"Domestic money remittances, telecom recharge, ...","₹136 Cr (FY ending Mar 2025, per Tracxn) / $31...",Uday Kumar (CEO) / Nandish Domlur (Founder & MD),Founder & CEO / Founder & Managing Director,Nandish Domlur and Uday Kumar co-founded Payis...,4.0,...,Agent-based financial inclusion services for m...,5.0,Reported ₹136 Cr annual revenue with 106 emplo...,40.0,High,Maybe,"Strong historical scale (₹136 Cr revenue, 28,0...","Payism's agent network of 28,000+ retailers se...","Tracxn (revenue, employee trend, deadpooled st...",27
5,FinMates Business Solutions Private Limited,finmates.in,Malad,Financial Services / B2B Professional Services,"CFO advisory, accounting/F&A outsourcing, taxa...",NaN,Pinkesh Jain,Founder & Lead CFO Service,Chartered Accountant (CA) and Company Secretar...,2.0,...,Virtual CFO / outsourced finance services mark...,5.0,LinkedIn shows active recent posting (2025) an...,32.0,Medium,Maybe,"Legitimate, verifiable professional services f...",Saw FinMates has been helping IT/Web3 startups...,Official website (finmates.in); LinkedIn (Pink...,26
6,Moneymax Fingrow Pvt.Ltd.,moneymaxfingrow.com,Mahalingapuram,Financial Services (Loan & Insurance Consulting),Financial consulting covering unsecured busine...,NaN,Dhejo and Mony,Co-Founders,19 years of combined experience across Banking...,2.0,...,Loan consulting/financial intermediation i